# Supporting checks

Regenerates the supporting checks of the paper from the derived data in `data/derived/` and the deposited files
`data/csv/mechanics.csv`, `data/csv/analysis_ree_samples.csv` and `data/csv/analysis_rg_samples.csv`. No simulation is
run and no raw simulation output is needed. How each derived file was made from the raw data is recorded in
`data/derived/README.md`.

Run from this folder:
`python -m nbconvert --to notebook --execute generate_checks.ipynb --output generate_checks_executed.ipynb`

Outputs go to `tables_checks/` (LaTeX tables and the statistics tables behind Figs. 6 and 7), `tables_checks/out/` (text
logs) and `figs_checks/` (figures). The `% source:` lines at the top of each `.tex` file name the inputs.

| Cell | Check | Outputs |
|---|---|---|
| Composition | composition of the CG networks | `table_composition.tex` |
| Generator benchmark | degree-distribution control and infeasible requests | `table_generator.tex`, `table_infeasible.tex` |
| Stationarity | equilibration stationarity of the CG ensemble | `fig_stationarity.png` |
| Orientation | lattice imprint on strand orientation | `fig_orientation.png` |
| Quadrants | counts and volumes per quadrant | `table_quadrants.tex` |
| Stress definition | true (Cauchy) versus nominal stress | `fig_true_vs_nominal.png`, `fig6_cells_summary_12pull.csv` |
| Variance decomposition | pull-to-pull versus network variance over four velocity seeds | `table_variance.tex`, `fig_variance_decomposition` |
| Fig. 7 statistics | Cohen's d, bootstrap intervals, Welch and adjusted p values; degree bands of Fig. 6b | `figure6_stats.csv`, `figure6_bars.csv`, `figure6_continuous.csv`, `figure5b_degree_bands_k12.csv` |
| Statistics tables | the same statistics as LaTeX tables, with the nominal-stress column | `table_cells.tex`, `table_continuous.tex`, `table_bars.tex` |
| Tg per cooling history | O-O bond check, Tg of each of the three cooling histories, MSD lag scan | `table_OO.tex`, `table_Tg.tex`, `table_lag.tex` |
| MSD lag | choice of the 4 ps MSD lag | `fig_Tg_lag_rationale` |
| Chain statistics | end-to-end distance and radius of gyration of the atomistic strands | `table_chain_stats.tex` |
| Software | software versions and random seeds | `table_software.tex` |

In [ ]:
# --- Setup ---
import warnings
# silence a harmless warning whose text carries local file paths (scipy 1.15.2 with numpy 2.5.2, see requirements.txt)
warnings.filterwarnings('ignore', message='A NumPy version', category=UserWarning)
import os, sys, re, json, itertools, collections
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import ttest_ind, skew, kurtosis
from scipy.optimize import curve_fit

DATA_PATH = 'data'
REV_PATH = os.path.join(DATA_PATH, 'derived')
FIG_DIR = 'figs_checks'                   # figures
TAB_DIR = 'tables_checks'                 # tables (.tex) and the statistics tables (.csv)
LOG_DIR = os.path.join(TAB_DIR, 'out')    # text logs
for d in (FIG_DIR, TAB_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.append(os.path.abspath(DATA_PATH))   # npj_style_v1


def require(path):
    """Fail loudly: every input path is relative to this notebook's folder."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required input not found: {path}. Run the notebook from its own folder (see README.md).")
    return path


def rev(*parts):
    return require(os.path.join(REV_PATH, *parts))


def read_csv_exact(path, **kwargs):
    """Parse floats exactly (pandas' default parser can be off in the last bit)."""
    return pd.read_csv(require(path), float_precision='round_trip', **kwargs)


def check_same_table(new, ref, name, tol=1e-12):
    """Non-float columns identical, float columns equal within tol (the original scripts parsed CSVs with pandas'
    default float parser, which can differ from the exact value in the last bit)."""
    assert list(new.columns) == list(ref.columns) and len(new) == len(ref), name
    fl = [c for c in ref.columns if ref[c].dtype.kind == 'f']; ot = [c for c in ref.columns if c not in fl]
    assert (new[ot].astype(str).values == ref[ot].astype(str).values).all(), name
    dmax = float(np.nanmax(np.abs(new[fl].to_numpy(float) - ref[fl].to_numpy(float)))) if fl else 0.0
    assert dmax < tol, (name, dmax)
    print(f"  {name:34s} vs packaged: non-float columns identical, max |float difference| {dmax:.1e}")


def load_pull_cube():
    """3,924 pulls -> dict of (327 networks x 3 axes x 4 velocity seeds) arrays, and the network order."""
    P = read_csv_exact(rev('mechanics', 'mechanics_pulls_4seeds.csv'))
    nets = list(dict.fromkeys(P['folder_name']))
    if len(P) != 12 * len(nets):
        raise ValueError('mechanics_pulls_4seeds.csv must hold 3 axes x 4 seeds per network')
    P = P.assign(_i=P['folder_name'].map({n: i for i, n in enumerate(nets)})).sort_values(['_i', 'axis', 'seed'])
    cube = {p: P[p].to_numpy().reshape(len(nets), 3, 4) for p in ('uts', 'toughness', 'sab')}
    cube['nets'] = np.array(nets)
    return cube, nets


print('output folders:', FIG_DIR, TAB_DIR, LOG_DIR)

In [ ]:
# --- Composition of the 327 CG networks ---
# The 2-core audit columns are in data/derived/cg_structure/network_2core_audit.csv.
M = read_csv_exact(os.path.join(DATA_PATH, 'csv', 'mechanics.csv')).set_index('folder_name')
F = read_csv_exact(rev('cg_structure', 'network_2core_audit.csv')).set_index('folder_name').reindex(M.index)
T2 = read_csv_exact(rev('cg_structure', 'task2_counts_per_network.csv')).set_index('folder_name').reindex(M.index)
out = []; P = out.append
n = len(M); P(f"networks: {n}")
cnt = M[[f'cnt_d{i}' for i in range(7)]].to_numpy(int)
assert (cnt.sum(1) == 216).all() and (M.total_nodes == 216).all()
E = (cnt * np.arange(7)).sum(1) / 2
Nact = 216 - cnt[:, 0]
assert np.allclose(E, F.E) and np.allclose(Nact, F.N_active)
assert np.allclose(E, T2.strands) and np.allclose(cnt[:, 0], T2.vacancies) and np.allclose(cnt[:, 1], T2.n_f1)
cyc = E - Nact + 1
assert np.allclose(cyc, M.cyclomatic_idx), "cycle rank != deposited cyclomatic_idx"
assert np.allclose(F.core_edges, T2.core_strands)
casc = Nact - F.core_nodes.to_numpy() - cnt[:, 1]
assert np.allclose(casc, F.cascade)
meandeg = 2 * E / Nact
assert np.allclose(meandeg, T2.mean_degree_occupied)
assert np.allclose(M.frac_dangling, cnt[:, 1] / 216)
P(f"frac_loops == 0 in all networks: {(M.frac_loops == 0).all()} (max {M.frac_loops.max()})")
P(f"2-core cascade removes more than the f=1 sites in {(casc > 0).sum()} of {n} networks; extra nodes {casc.min():.0f}-{casc.max():.0f}")
P(f"f=2 sites present in {(cnt[:, 2] > 0).sum()} networks; f=2 count {cnt[:, 2].min()}-{cnt[:, 2].max()}")
P(f"networks with no f=1 sites: {(cnt[:, 1] == 0).sum()}")
P(f"beads = junction beads + 30 x strands: {(T2.beads == (216 - cnt[:, 0]) + 30 * E).all()}  (beads {T2.beads.min()}-{T2.beads.max()})")
rows = [("Vacancies, $f=0$ (\\texttt{cnt\\_d0})", cnt[:, 0], 0),
        ("Sites with $f=1$ (dangling ends)", cnt[:, 1], 0),
        ("Sites with $f=2$ (chain extenders)", cnt[:, 2], 0),
        ("Sites with $f=3$", cnt[:, 3], 0), ("Sites with $f=4$", cnt[:, 4], 0),
        ("Sites with $f=5$", cnt[:, 5], 0), ("Sites with $f=6$", cnt[:, 6], 0),
        ("Active sites, $N_\\mathrm{active}$ ($f\\geq1$)", Nact, 0),
        ("Strands (graph edges), $E$", E, 0),
        ("2-core nodes", F.core_nodes.to_numpy(), 0), ("2-core edges", F.core_edges.to_numpy(), 0),
        ("Extra nodes removed by the 2-core cascade", casc, 0),
        ("Cycle rank, $E-N_\\mathrm{active}+1$", cyc, 0),
        ("Mean degree over active sites, $2E/N_\\mathrm{active}$", meandeg, 3),
        ("Dangling-end fraction ($f=1$ sites / 216)", cnt[:, 1] / 216, 3),
        ("Beads (junction beads $+\\,30E$)", T2.beads.to_numpy(), 0)]
L = [r"% source: data/csv/mechanics.csv (cnt_d0..cnt_d6, cyclomatic_idx, frac_dangling);",
     r"%         data/derived/cg_structure/network_2core_audit.csv (core_nodes, core_edges, cascade, E, N_active);",
     r"%         data/derived/cg_structure/task2_counts_per_network.csv (strands, beads; cross-check)",
     r"% generated by generate_checks.ipynb",
     r"\begin{tabular}{lrrr}", r"\toprule", r"Quantity (per network) & Min & Mean & Max \\", r"\midrule"]
for name, v, d in rows:
    v = np.asarray(v, float)
    fmt = (lambda x: f"{x:.{d}f}") if d else (lambda x: f"{x:.0f}")
    L.append(f"{name} & {fmt(v.min())} & {v.mean():.{max(d, 1)}f} & {fmt(v.max())} \\\\")
    P(f"{name:60s} min {v.min():.4f} mean {v.mean():.4f} max {v.max():.4f}")
L += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_composition.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")
open(os.path.join(LOG_DIR, 'table_composition_output.txt'), 'w', encoding='utf-8').write("\n".join(out) + "\n")
print("\n".join(out))

In [ ]:
# --- Generator benchmark tables ---
# The benchmark tables are read from data/derived/generator_benchmark/tables.md. The markdown tables are converted
# cell by cell; no number is retyped.
md = open(rev('generator_benchmark', 'tables.md'), encoding='utf-8').read().splitlines()


def tables(lines):
    out, cur = [], []
    for l in lines:
        if l.startswith('|'):
            cur.append(l)
        elif cur:
            out.append(cur); cur = []
    if cur: out.append(cur)
    return [[[c.strip() for c in row.strip().strip('|').split('|')] for row in t if not re.match(r'^\|\s*-', row)] for t in out]


def tex(s):
    """Escape one markdown cell for LaTeX; math fragments go through placeholders so later rules cannot touch them."""
    parts = re.split(r'(`[^`]*`)', s)
    res = []
    for p in parts:
        if p.startswith('`'):
            res.append(r'\texttt{' + p.strip('`').replace('_', r'\_').replace(',', r',\allowbreak{}') + '}'); continue
        store = []

        def keep(m_tex):
            store.append(m_tex); return f"@@{len(store)-1}@@"
        p = p.replace('lambda_2', keep(r'$\lambda_2$'))
        p = re.sub(r'(\d+)\s*<\s*k\s*<=\s*1e4', lambda m: keep(rf"${m.group(1)}<k\le 10^4$"), p)
        p = re.sub(r'k\s*<=\s*(\d+)', lambda m: keep(rf"$k\le {m.group(1)}$"), p)
        p = re.sub(r'k\s*>\s*1e4', lambda m: keep(r"$k>10^4$"), p)
        p = re.sub(r'k\s*>\s*10\^(\d+)', lambda m: keep(rf"$k>10^{{{m.group(1)}}}$"), p)
        p = re.sub(r'(\d+)\^(\d+)', lambda m: keep(rf"${m.group(1)}^{{{m.group(2)}}}$"), p)
        for a, b in (('<=', r'$\le$'), ('>=', r'$\ge$'), ('->', r'$\to$'), ('<', '$<$'), ('>', '$>$'), ('~', r'$\sim$'), ('+-', r'$\pm$')):
            if a in p:
                p = p.replace(a, keep(b))
        p = p.replace('\\', r'\textbackslash{}').replace('%', r'\%').replace('&', r'\&').replace('#', r'\#').replace('_', r'\_\allowbreak{}')
        p = re.sub(r'@@(\d+)@@', lambda m: store[int(m.group(1))], p)
        res.append(p)
    return ''.join(res).replace('$$', '')   # merge adjacent inline-math fragments


T = tables(md)
g1, g2 = T[0], T[1]
assert g1[0][0] == 'Method' and g2[0][0] == 'Request', (g1[0], g2[0])

L = [r"% source: data/derived/generator_benchmark/tables.md (Table G1)",
     r"% generated by generate_checks.ipynb",
     r"\begingroup\scriptsize\setlength{\tabcolsep}{3pt}\renewcommand{\arraystretch}{1.15}",
     r"\begin{longtable}{>{\raggedright\arraybackslash}p{3.3cm}>{\raggedright\arraybackslash}p{2.1cm}>{\raggedright\arraybackslash}p{2.4cm}>{\raggedright\arraybackslash}p{2.2cm}>{\raggedright\arraybackslash}p{1.3cm}>{\raggedright\arraybackslash}p{3.6cm}}",
     r"\caption{Degree-distribution control: success, runtime and error of the generator (Table G1 of the benchmark). SC $6\times6\times6$ periodic scaffold (216 sites, $z=6$), maximum $f=6$, unless stated. Error: $L_1$ distance between achieved and requested site counts over $f=0,\dots,6$; every returned graph was checked independently of the generator's own records. Times are wall clock per graph (median / 90th percentile / maximum) on one core of an Intel i9-14900 (Windows 11); the paper-era binary ran under WSL2. Target A = \texttt{10\_0\_26\_75\_43\_53\_9} (the documented example, no dangling ends); target B = \texttt{10\_44\_16\_35\_44\_13\_54} (largest deposited trial index, 575,056). Unused candidates: the 221 rows of the candidate list that are not among the 327 networks.}\label{tab:generator}\\",
     r"\toprule", " & ".join(r"\textbf{" + tex(c) + "}" for c in g1[0]) + r" \\", r"\midrule", r"\endfirsthead",
     r"\toprule", " & ".join(r"\textbf{" + tex(c) + "}" for c in g1[0]) + r" \\", r"\midrule", r"\endhead", r"\bottomrule", r"\endfoot"]
for row in g1[1:]:
    L.append(" & ".join(tex(c) for c in row) + r" \\")
L += [r"\end{longtable}", r"\endgroup"]
open(os.path.join(TAB_DIR, 'table_generator.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")

L = [r"% source: data/derived/generator_benchmark/tables.md (Table G2)",
     r"% generated by generate_checks.ipynb",
     r"\begingroup\scriptsize\setlength{\tabcolsep}{3pt}\renewcommand{\arraystretch}{1.15}",
     r"\begin{longtable}{>{\raggedright\arraybackslash}p{4.6cm}>{\raggedright\arraybackslash}p{2.4cm}>{\raggedright\arraybackslash}p{2.7cm}>{\raggedright\arraybackslash}p{2.3cm}>{\raggedright\arraybackslash}p{2.3cm}}",
     r"\caption{Infeasible or ill-posed requests and the generator's response (Table G2 of the benchmark). Base target A on SC $6\times6\times6$ unless stated; the request is given as \texttt{f:count} pairs. Outcome and time to answer; strict runs capped at 60~s.}\label{tab:infeasible}\\",
     r"\toprule", " & ".join(r"\textbf{" + tex(c) + "}" for c in g2[0]) + r" \\", r"\midrule", r"\endfirsthead",
     r"\toprule", " & ".join(r"\textbf{" + tex(c) + "}" for c in g2[0]) + r" \\", r"\midrule", r"\endhead", r"\bottomrule", r"\endfoot"]
for row in g2[1:]:
    cells_ = [tex(c) for c in row]
    if row[0].startswith('two_f1_only'):
        cells_[1] = cells_[1] + r"$^{a}$"
    L.append(" & ".join(cells_) + r" \\")
L += [r"\end{longtable}",
      r"\vspace{-0.6\baselineskip}\noindent{\scriptsize $^{a}$Listed as scaffold-unreachable in the benchmark report. The request corresponds to a single strand between two adjacent sites, which the SC scaffold admits; the exact search returned an error after 9.6~s. The row therefore illustrates that an unsuccessful search is not a proof of infeasibility.}",
      r"\endgroup"]
open(os.path.join(TAB_DIR, 'table_infeasible.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")
print(len(g1) - 1, "rows in G1;", len(g2) - 1, "rows in G2")

In [ ]:
# --- Equilibration stationarity (ensemble panels) ---
# Plotting code of scripts/raw_data_analyses/cg_structure/task3_stationarity.py (figure part). The plotted quantities
# (block means, drift t statistics, block-5 averages and their standard errors, snapshot shear stresses, thermal SD) are
# read from data/derived/cg_structure/stationarity_per_network.csv, computed with that script's own code from the 55 MB
# thermo cache.
st = read_csv_exact(rev('cg_structure', 'stationarity_per_network.csv'))
with open(rev('cg_structure', 'stationarity_scalars.json')) as fh:
    box_stats = json.load(fh)
mech = read_csv_exact(rev('mechanics', 'mechanics_12pull.csv'))[["folder_name", "uts_mean", "toughness_mean", "Quadrant"]]
assert list(st.folder_name) == list(mech.folder_name)
NN = len(st)
bm = {k: st[[f"{k}_block{b}_mean" for b in range(1, 6)]].to_numpy() for k in ("pe", "density", "box_aniso")}   # (327, 5)
per = st
tc18 = stats.t.ppf(0.975, 18)
shear = st[["final_pxy_snapshot", "final_pxz_snapshot", "final_pyz_snapshot"]].values
tsd = st["thermal_sd_halfdev"].to_numpy().mean()
var_ratio_shear = (shear ** 2).mean() / tsd ** 2
n_sh = shear.size
ci_lo = n_sh * var_ratio_shear / stats.chi2.ppf(0.975, n_sh); ci_hi = n_sh * var_ratio_shear / stats.chi2.ppf(0.025, n_sh)
uts = mech.uts_mean.mean()

matplotlib.rcdefaults()   # start from matplotlib defaults, as the standalone script did
plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
                     "xtick.major.width": 0.6, "ytick.major.width": 0.6, "savefig.dpi": 300})
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK2 = "#52514e"

# ensemble panels
fig, axs = plt.subplots(2, 3, figsize=(7.4, 5.2))
xb = np.arange(1, 6)


def band(axx, y, col, lab=True):
    for i in range(NN):
        axx.plot(xb, y[i], color="0.8", lw=0.3, alpha=0.5, zorder=1)
    lo, hi = np.quantile(y, [0.05, 0.95], axis=0)
    axx.fill_between(xb, lo, hi, color=col, alpha=0.18, lw=0, zorder=2, label="5-95%" if lab else None)
    axx.plot(xb, np.median(y, 0), color=col, lw=1.6, marker="o", ms=3.5, zorder=3, label="median" if lab else None)
    axx.set_xticks(xb); axx.set_xlabel(r"Production block (4000 $\tau$ each)")


for axx, k, lab in ((axs[0, 0], "pe", "a  PE per bead - network mean"),
                    (axs[0, 1], "density", r"b  density - network mean ($\sigma^{-3}$)")):
    y = bm[k] - bm[k].mean(1, keepdims=True)
    band(axx, y, C1, lab=(k == "pe"))
    axx.axhline(0, color="k", lw=0.5)
    axx.set_title(lab, fontsize=8, loc="left")
axs[0, 0].legend(frameon=False, fontsize=6.5, loc="lower left", ncol=2)
axx = axs[0, 2]
band(axx, bm["box_aniso"], C2, lab=False)
axx.axhline(box_stats["anneal_end_aniso_mean"], color="k", ls=":", lw=0.8)
axx.text(5, 0.42, "dotted: anneal end (mean)", fontsize=6, color=INK2, ha="right")
axx.set_title("c  box shape: max(L)/min(L) - 1", fontsize=8, loc="left")
axx = axs[1, 0]
tt = np.linspace(-6, 6, 300)
for k, col, nm in (("pe", C1, "PE"), ("density", C2, "density"), ("press", C3, "P")):
    axx.hist(per[f"{k}_t20"], bins=np.linspace(-6, 6, 41), histtype="step", density=True, color=col, lw=1.1,
             label=f"{nm} ({np.mean(np.abs(per[f'{k}_t20']) > tc18):.0%})")
axx.plot(tt, stats.t.pdf(tt, 18), color="k", lw=0.8, ls="--", label="t(18) null (5%)")
axx.set_xlabel("Drift t (OLS on 20 sub-block means)"); axx.set_ylabel("Density")
axx.set_ylim(0, 0.62)
axx.legend(frameon=False, fontsize=6, loc="upper left", ncol=2, title=r"share $|t| > t_{0.975}$", title_fontsize=6)
axx.set_title("d  drift tests, 327 networks", fontsize=8, loc="left")
axx = axs[1, 1]
bins = np.linspace(-0.04, 0.04, 33)
axx.hist(per.press_block5_mean, bins=bins, histtype="stepfilled", color=C1, alpha=0.35, density=True, label="P")
dev_all = np.concatenate([per[f"{k}_block5_mean"] for k in ["dev_xy", "dev_yz", "dev_zx"]])
axx.hist(dev_all, bins=bins, histtype="step", color=C2, lw=1.1, density=True, label=r"$P_{xx}-P_{yy}$ etc.")
xx = np.linspace(-0.04, 0.04, 200)
axx.plot(xx, stats.norm.pdf(xx, 0, per.press_block5_se.mean()), color=C1, ls="--", lw=0.8, label="sampling SE, P")
axx.plot(xx, stats.norm.pdf(xx, 0, per.dev_xy_block5_se.mean()), color=C2, ls="--", lw=0.8, label="sampling SE, diff.")
axx.set_ylim(0, 1.45 * axx.get_ylim()[1])
axx.set_xlabel(r"Block-5 time average ($\epsilon\sigma^{-3}$)"); axx.set_ylabel("Density")
axx.legend(frameon=False, fontsize=6, loc="upper left", ncol=2)
axx.set_title(f"e  residual stress (mean UTS = {uts:.2f})", fontsize=8, loc="left")
axx = axs[1, 2]
bins = np.linspace(-0.4, 0.4, 41)
axx.hist(shear.ravel(), bins=bins, density=True, histtype="stepfilled", color=C3, alpha=0.4,
         label=r"$P_{xy},P_{xz},P_{yz}$ (3x327)")
xx = np.linspace(-0.4, 0.4, 300)
axx.plot(xx, stats.norm.pdf(xx, 0, tsd), color="k", lw=0.8, ls="--", label=f"thermal N(0, {tsd:.3f})")
axx.set_ylim(0, 1.35 * axx.get_ylim()[1])
axx.set_xlabel("Shear stress, pull start state"); axx.set_ylabel("Density")
axx.legend(frameon=False, fontsize=6, loc="upper left")
axx.set_title(f"f  shear <x²>/thermal {var_ratio_shear:.2f} [{ci_lo:.2f}, {ci_hi:.2f}]", fontsize=7.2, loc="left")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig_stationarity.png"))
plt.close(fig)
print(f"stationarity figure written; shear <x^2>/thermal = {var_ratio_shear:.2f} [{ci_lo:.2f}, {ci_hi:.2f}]")

In [ ]:
# --- Lattice imprint on strand orientation ---
# Plotting code of scripts/raw_data_analyses/cg_structure/task1_figure.py. The strand-level inputs are the
# columns/keys of the 55 MB cache that the figure uses (strand_memory_eq.npz, strand_vectors_first_eq.npz).
df = read_csv_exact(rev('cg_structure', 'task1_orientation_per_network.csv'))
ax = read_csv_exact(rev('cg_structure', 'task1_per_axis.csv'))
S = pd.DataFrame(dict(np.load(rev('cg_structure', 'strand_memory_eq.npz'))))
vz = np.load(rev('cg_structure', 'strand_vectors_first_eq.npz'))
corr = read_csv_exact(rev('cg_structure', 'task1_mechanics_correlations.csv'))
wc = read_csv_exact(rev('cg_structure', 'task1_within_network_axis_correlations.csv'))

matplotlib.rcdefaults()   # start from matplotlib defaults, as the standalone script did
plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
                     "savefig.dpi": 300})
C1, C2, C3, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
INK2 = "#52514e"
fig, axs = plt.subplots(2, 3, figsize=(7.4, 5.3))

# (a) build progression
a = axs[0, 0]
lab = ["gen", "min2", "min3", "minsoft", "minfin", "minreal", "nvt", "first", "eq"]
short = ["generator", "min2", "min3", "soft min", "LJ min", "FENE min", "NVT", "MD start", "equilibrated"]
xs = np.arange(len(lab))
for col, key, nm in ((C1, "str_C4", r"$C_4=\langle u_x^4+u_y^4+u_z^4\rangle$"), (C2, "str_P2mem", r"memory $\langle P_2(u\cdot e_0)\rangle$")):
    v = np.array([df[f"{s}_{key}"].values for s in lab])
    a.fill_between(xs, np.quantile(v, .05, 1), np.quantile(v, .95, 1), color=col, alpha=0.2, lw=0)
    a.plot(xs, np.median(v, 1), color=col, marker="o", ms=3, lw=1.2, label=nm)
a.axhline(0.6, color=C1, ls=":", lw=0.8); a.axhline(0, color=C2, ls=":", lw=0.8)
a.text(0.1, 0.63, "isotropic 3/5", fontsize=6, color=C1)
a.text(0.1, 0.03, "no memory", fontsize=6, color=C2)
a.set_xticks(xs); a.set_xticklabels(short, rotation=55, ha="right", fontsize=6.3)
a.set_ylabel("Strands: median, 5-95% of networks")
a.set_ylim(-0.05, 1.12)
a.legend(frameon=False, fontsize=6.2, loc="lower left", bbox_to_anchor=(0.0, 0.12))
a.set_title("a  orientation through the build", fontsize=8, loc="left")

# (b) z-scores vs iid isotropic null (equilibrated)
b = axs[0, 1]
bins = np.linspace(-4, 6, 41); zz = np.linspace(-4, 4, 200)
for col, cz, nm, side in ((C1, "eq_str_zC4_iid", r"$C_4$ strands", "gt"), (C3, "eq_str_zS_iid", r"$S$ strands", "gt"),
                          (C5, "eq_bond_zC4_iid", r"$C_4$ bonds", "abs")):
    frac = np.mean(df[cz] > 2) if side == "gt" else np.mean(np.abs(df[cz]) > 2)
    b.hist(df[cz], bins=bins, density=True, histtype="step", color=col, lw=1.1,
           label=f"{nm} ({frac:.0%} z>2)" if side == "gt" else f"{nm} ({frac:.0%} |z|>2)")
b.plot(zz, stats.norm.pdf(zz), "k--", lw=0.8, label="N(0,1)")
b.set_ylim(0, 0.75)
b.set_xlabel("z vs isotropic null (same n), equilibrated"); b.set_ylabel("Density")
b.legend(frameon=False, fontsize=6, loc="upper left")
b.set_title("b  cubic and second-rank order", fontsize=8, loc="left")

# (c) per-strand memory, pooled
c = axs[0, 2]
edges = np.linspace(0, 1, 21); mid = 0.5 * (edges[1:] + edges[:-1])
for st_, col, nm in (("first", C2, "MD start"), ("eq", C1, "equilibrated")):
    cc = []
    for n in df.folder_name:
        R = vz[f"{st_}__{n}"].astype(float); u = R / np.linalg.norm(R, axis=1)[:, None]
        cc.append(np.abs(u[np.arange(len(u)), vz[f"axis__{n}"].astype(int)]))
    cc = np.concatenate(cc)
    h, _ = np.histogram(cc, bins=edges, density=True)
    c.step(mid, h, where="mid", color=col, lw=1.2, label=f"{nm}: " + r"$\langle P_2\rangle$" + f"={np.mean(1.5 * cc ** 2 - 0.5):.2f}")
c.axhline(1, color="k", ls="--", lw=0.8, label="no memory (uniform)")
c.set_xlabel(r"$|u\cdot e_0|$, strand vs its scaffold direction"); c.set_ylabel("Probability density (all strands)")
c.legend(frameon=False, fontsize=6.2, loc="upper left")
c.set_title("c  per-strand directional memory", fontsize=8, loc="left")

# (d) memory vs end-junction degrees
d = axs[1, 0]
g = S.groupby(["fmin", "fmax"]).agg(n=("P2", "size"), P2=("P2", "mean"), sd=("P2", "std")).reset_index()
cols = {1: "0.55", 2: C3, 3: C1, 4: C5, 5: C2, 6: "k"}
for fmin in range(1, 7):
    gg = g[(g.fmin == fmin) & (g.n >= 200)]
    if len(gg):
        d.errorbar(gg.fmax, gg.P2, yerr=gg.sd / np.sqrt(gg.n), color=cols[fmin], marker="o", ms=3, lw=1, capsize=0,
                   label=f"lower-degree end f={fmin}")
d.axhline(0, color="k", ls=":", lw=0.8)
d.set_xlabel("Degree of the higher-degree end junction"); d.set_ylabel(r"$\langle P_2(u\cdot e_0)\rangle$, equilibrated")
d.legend(frameon=False, fontsize=5.8, loc="upper left")
d.set_title("d  memory is set by junction pinning", fontsize=8, loc="left")

# (e) C4 vs UTS
e = axs[1, 1]
e.scatter(df.eq_str_C4, df.uts_mean, s=6, color=C1, alpha=0.6, lw=0)
r = corr[(corr.x == "C4 strands (eq)") & (corr.y == "UTS")].iloc[0]
e.axvline(0.6, color="k", ls=":", lw=0.8)
e.set_xlabel(r"$C_4$ strands, equilibrated"); e.set_ylabel("Network-mean UTS (12 pulls)")
e.set_title(f"e  rho = {r.rho:+.2f}; partial|degree counts -0.16", fontsize=7.2, loc="left")

# (f) within-network per-axis: 2-core weakest lattice plane vs UTS_a
f = axs[1, 2]
dd = ax.copy()
for col in ("core_plane_min", "uts", "eq_Qaa"):
    dd[col + "_c"] = dd[col] - dd.groupby("folder_name")[col].transform("mean")
jit = np.random.default_rng(1).uniform(-0.25, 0.25, len(dd))
f.scatter(dd.core_plane_min_c + jit, dd.uts_c, s=4, color=C5, alpha=0.45, lw=0)
w1 = wc[(wc.x == "2-core min plane count along a") & (wc.y == "UTS_a")].iloc[0]
w2 = wc[(wc.x == "Q_aa strands (eq)") & (wc.y == "UTS_a")].iloc[0]
f.set_xlabel("2-core strands crossing the weakest\nlattice plane normal to a (centred)")
f.set_ylabel(r"UTS$_a$, 4-seed mean (centred)")
f.set_title(f"f  per-axis: r = {w1.r_within:+.2f}; " + r"$Q_{aa}$" + f": r = {w2.r_within:+.2f}", fontsize=7.2, loc="left")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig_orientation.png"))
plt.close(fig)
print("orientation figure written")

In [ ]:
# --- Counts and volumes per quadrant ---
# Per-quadrant part of scripts/raw_data_analyses/cg_structure/task2_counts.py, from the per-network counts it wrote.
# The table is recomputed here and checked against the packaged task2_quadrant_table.csv.
dfc = read_csv_exact(rev('cg_structure', 'task2_counts_per_network.csv'))
VARS = [("vacancies", "Vacancies (f=0 sites)", "{:.1f}"), ("n_f1", "Dangling ends (f=1)", "{:.1f}"),
        ("n_f2", "Chain extenders (f=2)", "{:.1f}"), ("n_f3plus", "Junctions (f>=3)", "{:.1f}"),
        ("strands", "Strands (edges)", "{:.1f}"), ("core_strands", "2-core strands", "{:.1f}"),
        ("core_junctions_f3plus", "2-core junctions (f>=3 in core)", "{:.1f}"),
        ("mean_degree_occupied", "Mean degree (occupied sites)", "{:.3f}"),
        ("beads", "Beads", "{:.0f}"), ("V_lastblock_mean", "Volume <V> (sigma^3)", "{:.0f}"),
        ("number_density", "Bead density (sigma^-3)", "{:.4f}"),
        ("junction_density_f3plus", "Junction density f>=3 (1e-3 sigma^-3)", "{:.3f}"),
        ("core_strand_density", "2-core strand density (1e-3 sigma^-3)", "{:.2f}")]
scale = {"junction_density_f3plus": 1e3, "core_strand_density": 1e3}
qs = ["Q1: Weak/Brittle", "Q2: Weak/Tough", "Q3: Strong/Brittle", "Q4: Strong/Tough"]
tab = []
for v, lab_, fmt in VARS:
    x = dfc[v] * scale.get(v, 1.0)
    row = {"variable": lab_}
    for q in qs:
        s = x[dfc.Quadrant == q]
        row[f"{q[:2]}_mean"] = s.mean(); row[f"{q[:2]}_sd"] = s.std(); row[f"{q[:2]}_min"] = s.min(); row[f"{q[:2]}_max"] = s.max()
        row[f"{q[:2]}_n"] = len(s)
    row["KW_p"] = stats.kruskal(*[x[dfc.Quadrant == q] for q in qs]).pvalue
    for m, mn in (("uts_mean", "UTS"), ("toughness_mean", "T")):
        rr = stats.spearmanr(x, dfc[m])
        row[f"rho_{mn}"] = rr.statistic; row[f"p_{mn}"] = rr.pvalue
    tab.append(row)
Q = pd.DataFrame(tab)
Q.to_csv(os.path.join(LOG_DIR, 'task2_quadrant_table.csv'), index=False)
ref = read_csv_exact(rev('cg_structure', 'task2_quadrant_table.csv'))
num = [c for c in ref.columns if c != 'variable']
assert (ref.variable == Q.variable).all() and np.allclose(ref[num].to_numpy(float), Q[num].to_numpy(float), rtol=1e-12, atol=0)
print(f"quadrant table vs packaged task2_quadrant_table.csv: max |difference| "
      f"{np.nanmax(np.abs(ref[num].to_numpy(float) - Q[num].to_numpy(float))):.1e}")


def fp(p):
    if p < 1e-3:
        m, e = f"{p:.0e}".split('e'); return f"$10^{{{int(e)}}}$" if m == '1' else f"${m}\\times10^{{{int(e)}}}$"
    return f"{p:.2f}"


lab = {'Vacancies (f=0 sites)': 'Vacancies ($f=0$)', 'Dangling ends (f=1)': 'Dangling ends ($f=1$)', 'Chain extenders (f=2)': 'Chain extenders ($f=2$)',
       'Junctions (f>=3)': r'Junctions ($f\geq3$)', 'Strands (edges)': 'Strands (edges)', '2-core strands': 'Elastically active strands (2-core edges)',
       '2-core junctions (f>=3 in core)': r'2-core junctions ($f\geq3$ in core)', 'Mean degree (occupied sites)': 'Mean degree (active sites)',
       'Beads': 'Beads', 'Volume <V> (sigma^3)': r'Volume $\langle V\rangle$ ($\sigma^3$)', 'Bead density (sigma^-3)': r'Bead density ($\sigma^{-3}$)',
       'Junction density f>=3 (1e-3 sigma^-3)': r'Junction density, $f\geq3$ ($10^{-3}\sigma^{-3}$)',
       '2-core strand density (1e-3 sigma^-3)': r'2-core strand density ($10^{-3}\sigma^{-3}$)'}
dec = {'Mean degree (occupied sites)': 3, 'Bead density (sigma^-3)': 4, 'Junction density f>=3 (1e-3 sigma^-3)': 2, '2-core strand density (1e-3 sigma^-3)': 2,
       'Beads': 0, 'Volume <V> (sigma^3)': 0}
n = [int(Q[f'Q{q}_n'].iloc[0]) for q in (1, 2, 3, 4)]
L = [r"% source: data/derived/cg_structure/task2_quadrant_table.csv (task2_counts.py)",
     r"% generated by generate_checks.ipynb",
     r"\begin{tabular}{lccccccc}", r"\toprule",
     f"Quantity & Q1 ($n={n[0]}$) & Q2 ($n={n[1]}$) & Q3 ($n={n[2]}$) & Q4 ($n={n[3]}$) & KW $p$ & $\\rho$(UTS) & $\\rho$(T) \\\\", r"\midrule"]
for _, r in Q.iterrows():
    d = dec.get(r.variable, 1)
    cells_ = [f"{r[f'Q{q}_mean']:.{d}f} $\\pm$ {r[f'Q{q}_sd']:.{d}f}" for q in (1, 2, 3, 4)]
    L.append(f"{lab[r.variable]} & " + " & ".join(cells_) + f" & {fp(r.KW_p)} & ${r.rho_UTS:+.2f}$ & ${r.rho_T:+.2f}$ \\\\")
L += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_quadrants.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")
print('quadrant table written')

In [ ]:
# --- True (Cauchy) versus nominal stress ---
# Plotting code of scripts/raw_data_analyses/stress_definition/s3_figure.py. The box curves of the two example
# networks (panels a, b) are the 6 of 981 arrays of box_curves_seed1.npz that the figure uses; the unused per-pull table is
# not read. Also writes the Fig. 7 robustness summary fig6_cells_summary_12pull.csv (used as a cross-check by the cell table).
BLUE, ORANGE, AQUA, GRAY = '#2a78d6', '#eb6834', '#1baf7a', '#9a9993'
INK, INK2 = '#0b0b0b', '#52514e'
matplotlib.rcdefaults()   # start from matplotlib defaults, as the standalone script did
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2, 'ytick.color': INK2,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.titlesize': 10, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left'})

NET = read_csv_exact(rev('stress_definition', 'network_true_vs_nominal.csv'))
F6 = read_csv_exact(rev('stress_definition', 'fig6_cells_true_vs_nominal.csv'))
U = read_csv_exact(rev('stress_definition', 'area_universality_seed1.csv'))
z = np.load(rev('stress_definition', 'box_curves_seed1_examples.npz'))

# ---------------------------------------------------------------- Figure-6 robustness summary (12-pull)
p = F6[F6.set == '12pull'].set_index(['metric', 'transition'])
CS = p[['d_true', 'q_true', 'sig_true', 'd_nom', 'q_nom', 'sig_nom']].copy()
for v in ('12pull_all_incomp', '12pull_all_uts'):
    q = F6[F6.set == v].set_index(['metric', 'transition']); tag = v.replace('12pull_', '')
    CS[f'd_nom_{tag}'] = q.d_nom; CS[f'q_nom_{tag}'] = q.q_nom; CS[f'sig_nom_{tag}'] = q.sig_nom
nomsig = CS[['sig_nom', 'sig_nom_all_incomp', 'sig_nom_all_uts']]
CS['class'] = np.select([CS.sig_true & nomsig.all(axis=1), CS.sig_true & ~nomsig.any(axis=1), ~CS.sig_true & nomsig.all(axis=1), ~CS.sig_true & ~nomsig.any(axis=1)],
                        ['significant under both', 'true only', 'nominal only', 'neither'], 'depends on replicate area treatment')
CS.reset_index().to_csv(os.path.join(TAB_DIR, 'fig6_cells_summary_12pull.csv'), index=False)
check_same_table(read_csv_exact(os.path.join(TAB_DIR, 'fig6_cells_summary_12pull.csv')),
                 read_csv_exact(rev('stress_definition', 'fig6_cells_summary_12pull.csv')), 'fig6_cells_summary_12pull.csv')
print(CS['class'].value_counts().to_string())

# ---------------------------------------------------------------- example networks (12-pull quadrants)
pr = lambda x: stats.rankdata(x) / len(x)
NET['dUTSrank'] = pr(NET.UTS_nom_12pull) - pr(NET.UTS_true_12pull)
exA = NET[(NET.Q_true_12pull == 4) & (NET.Q_nom_12pull == 2)].sort_values('dUTSrank').folder_name.iloc[0]
exB = NET[(NET.Q_true_12pull == 1) & (NET.Q_nom_12pull == 3)].sort_values('dUTSrank').folder_name.iloc[-1]
print('\nexamples:', exA, '(Q4 -> Q2)', exB, '(Q1 -> Q3)')

fig, ax = plt.subplots(2, 4, figsize=(18, 8.4)); ax = ax.ravel()
for k, (col, lab) in enumerate(((2, 'True stress $-P_{aa}$ (current area)'), (None, 'Nominal stress $F/A_0$'))):
    a = ax[k]
    for fn, c, tag in ((exA, BLUE, 'A: Q4 true -> Q2 nominal'), (exB, ORANGE, 'B: Q1 true -> Q3 nominal')):
        for i, axn in enumerate('xyz'):
            arr = z[f'{fn}|{axn}']; e = arr[:, 1]; s = arr[:, 2] if col else arr[:, 2] * arr[:, 3]
            a.plot(e, s, color=c, lw=1.2, alpha=0.85, label=tag if i == 0 else None)
    a.set_xlabel('Engineering strain'); a.set_ylabel(lab + ', LJ units'); a.set_xlim(0, 18)
    a.set_title(('a  ' if k == 0 else 'b  ') + ('True stress, seed-1 pulls (3 axes each)' if k == 0 else 'Same pulls, nominal stress (measured area)'))
    a.axhline(0, color=GRAY, lw=0.6); a.legend(frameon=False, loc='upper left')

# c: volume ratio of pre-break states
a = ax[2]
a.fill_between(U.strain, U.VV0_intact_p5, U.VV0_intact_p95, color=BLUE, alpha=0.18, lw=0, label='5-95% of pulls not yet broken')
a.plot(U.strain, U.VV0_intact_mean, color=BLUE, lw=2, label='mean, pulls not yet broken')
a.plot(U.strain, U.VV0_p95, color=GRAY, lw=1, ls='--', label='95th pct, all pulls (incl. broken)')
a.axhline(1, color=INK2, lw=0.8, ls=':'); a.set_ylim(0.95, 1.8); a.set_xlabel('Engineering strain'); a.set_ylabel('$V/V_0 = (A/A_0)(1+\\varepsilon)$')
a.set_title('c  Volume is conserved to 0.5% up to strain 6'); a.legend(frameon=False, loc='upper left')
for _, r in U[U.strain.round(0).isin([10, 13])].iterrows():
    a.annotate(f"n={int(r.n_intact)}", (r.strain, r.VV0_intact_p95), textcoords='offset points', xytext=(0, 4), ha='center', color=INK2, fontsize=8)

# d, e: network means (12 pulls)
chg = (NET.Q_true_12pull != NET.Q_nom_12pull).to_numpy()
for k, (m, lab) in enumerate((('UTS', 'UTS'), ('tough', 'Toughness'))):
    a = ax[3 + k]; x = NET[f'{m}_true_12pull']; y = NET[f'{m}_nom_12pull']
    a.scatter(x[~chg], y[~chg], s=16, color=GRAY, alpha=0.7, lw=0, label=f'same quadrant (n={int((~chg).sum())})')
    a.scatter(x[chg], y[chg], s=16, color=ORANGE, alpha=0.9, lw=0, label=f'changes quadrant (n={int(chg.sum())})')
    a.axvline(np.median(x), color=INK2, lw=0.7, ls=':'); a.axhline(np.median(y), color=INK2, lw=0.7, ls=':')
    a.set_xlabel(f'{lab}, true stress (12-pull mean)'); a.set_ylabel(f'{lab}, nominal stress (12-pull mean)')
    a.set_title(f"{'d' if k == 0 else 'e'}  {lab}: Spearman {stats.spearmanr(x, y)[0]:.2f}; dotted = medians"); a.legend(frameon=False, loc='upper left', fontsize=8)


# f, h: quadrant transition matrices
def mat(a, qt, qn, title):
    M_ = pd.crosstab(qt, qn).reindex(index=[1, 2, 3, 4], columns=[1, 2, 3, 4], fill_value=0).to_numpy()
    a.imshow(M_, cmap=matplotlib.colors.LinearSegmentedColormap.from_list('b', ['#f4f8fd', '#86b6ef', '#184f95']), vmin=0)
    for i in range(4):
        for j in range(4):
            a.text(j, i, M_[i, j], ha='center', va='center', color='white' if M_[i, j] > 0.55 * M_.max() else INK, fontsize=10)
    names = ['Q1 weak/brittle', 'Q2 weak/tough', 'Q3 strong/brittle', 'Q4 strong/tough']
    a.set_xticks(range(4)); a.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=8); a.set_yticks(range(4)); a.set_yticklabels(names, fontsize=8)
    a.set_xlabel('Nominal-stress quadrant'); a.set_ylabel('True-stress quadrant'); a.set_title(title)
    for s in a.spines.values(): s.set_visible(False)


mat(ax[5], NET.Q_true_12pull, NET.Q_nom_12pull, f"f  12-pull means: {int(chg.sum())}/327 change ({100 * chg.mean():.0f}%)")
c3 = (NET.Q_true_seed1_3pull != NET.Q_nom_seed1_3pull)
mat(ax[7], NET.Q_true_seed1_3pull, NET.Q_nom_seed1_3pull, f"h  Seed 1 only, 3-pull means: {int(c3.sum())}/327 ({100 * c3.mean():.0f}%)")

# g: Figure-6 Cohen's d, true vs nominal (12-pull, primary nominal)
a = ax[6]; G = F6[F6.set == '12pull']
cats = [('BH-significant under both', G.sig_true & G.sig_nom, BLUE), ('true only', G.sig_true & ~G.sig_nom, ORANGE),
        ('nominal only', ~G.sig_true & G.sig_nom, AQUA), ('neither', ~G.sig_true & ~G.sig_nom, GRAY)]
lim = float(np.ceil(10 * (max(abs(G.d_true).max(), abs(G.d_nom).max()) + 0.05)) / 10)
a.plot([-lim, lim], [-lim, lim], color=INK2, lw=0.7, ls=':'); a.axhline(0, color=GRAY, lw=0.5); a.axvline(0, color=GRAY, lw=0.5)
for lab, m, c in cats:
    a.scatter(G.d_true[m], G.d_nom[m], s=30, color=c, lw=0.8, edgecolor='white', label=f'{lab} ({int(m.sum())})')
a.set_xlim(-lim, lim); a.set_ylim(-lim, lim); a.set_aspect('equal')
a.set_xlabel("Cohen's d, true-stress quadrants"); a.set_ylabel("Cohen's d, nominal-stress quadrants")
a.set_title(f"g  Figure-6 cells (55): r = {stats.pearsonr(G.d_true, G.d_nom)[0]:.2f}"); a.legend(frameon=False, loc='upper left', fontsize=8)

fig.suptitle('True (Cauchy) vs nominal (engineering) stress, 327 CG networks: seed 1 uses the measured box area; replicate seeds use A/A0 = 1/(1+strain)',
             x=0.01, ha='left', fontsize=11, color=INK)
fig.tight_layout(rect=(0, 0, 1, 0.96)); fig.savefig(os.path.join(FIG_DIR, 'fig_true_vs_nominal.png'), dpi=170); plt.close(fig)
print('true versus nominal stress figure written')

In [ ]:
# --- Four-seed variance decomposition (table and figure) ---
# The pull cube is rebuilt from data/derived/mechanics/mechanics_pulls_4seeds.csv (bitwise equal to four_seed_cube.npz,
# the output of scripts/raw_data_analyses/cg_mechanics/four_seed_analysis.py).
cube, nets = load_pull_cube()
out = []; P = out.append
P(f"cube shape {cube['uts'].shape}; pulls {cube['uts'].size}")


def decompose(Y):
    n, k, m = Y.shape; gm = Y.mean(); net = Y.mean((1, 2))[:, None, None]; axm = Y.mean((0, 2))[None, :, None]; cell = Y.mean(2)[:, :, None]
    SS_net = k * m * ((net - gm) ** 2).sum(); SS_ax = n * m * ((axm - gm) ** 2).sum(); SS_int = m * ((cell - net - axm + gm) ** 2).sum(); SS_err = ((Y - cell) ** 2).sum()
    MS_net, MS_ax, MS_int, MS_err = SS_net / (n - 1), SS_ax / (k - 1), SS_int / ((n - 1) * (k - 1)), SS_err / (n * k * (m - 1))
    return dict(v_net=max((MS_net - MS_int) / (k * m), 0), v_ax=max((MS_ax - MS_int) / (n * m), 0), v_int=max((MS_int - MS_err) / m, 0), v_err=MS_err,
                F_int=MS_int / MS_err, F_ax=MS_ax / MS_int, F_net=MS_net / MS_int)


def rel(d, k, m): return d['v_net'] / (d['v_net'] + d['v_int'] / k + d['v_err'] / (k * m))


def rel_fixed(d, k, m): return (d['v_net'] + d['v_int'] / k) / (d['v_net'] + d['v_int'] / k + d['v_err'] / (k * m))


def quadrants(u, t): return 1 + (t >= np.median(t)).astype(int) + 2 * (u >= np.median(u)).astype(int)


res = {}
for p in ('uts', 'toughness'):
    d4 = decompose(cube[p]); d3 = decompose(cube[p][:, :, :3]); tot = d4['v_net'] + d4['v_ax'] + d4['v_int'] + d4['v_err']
    fr = {c: d4[c] / tot for c in ('v_net', 'v_ax', 'v_int', 'v_err')}
    res[p] = dict(d4=d4, fr=fr, rel12=rel(d4, 3, 4), rel9=rel(d3, 3, 3), rel1=rel(d4, 1, 1), rel3=rel(d4, 3, 1),
                  relinf=d4['v_net'] / (d4['v_net'] + d4['v_int'] / 3),
                  relF12=rel_fixed(d4, 3, 4), relF3=rel_fixed(d4, 3, 1))
    P(f"{p}: fractions network {100*fr['v_net']:.1f}%  axis {100*fr['v_ax']:.1f}%  network x axis {100*fr['v_int']:.1f}%  seed {100*fr['v_err']:.1f}%"
      f"  | F(net x axis vs seed) {d4['F_int']:.2f}  F(axis) {d4['F_ax']:.2f}")
    P(f"   reliability, axes random: 1 pull {res[p]['rel1']:.3f}; 3 axes x 1 seed {res[p]['rel3']:.3f}; 9 pulls (3 seeds) {res[p]['rel9']:.3f}; 12 pulls {res[p]['rel12']:.3f}; ceiling (infinite seeds) {res[p]['relinf']:.3f}")
    P(f"   reliability, axes fixed:  3 axes x 1 seed {res[p]['relF3']:.3f}; 12 pulls {res[p]['relF12']:.3f}")
    P(f"   per-seed grand mean {np.round(cube[p].mean((0, 1)), 4)}")

u12, t12 = cube['uts'].mean((1, 2)), cube['toughness'].mean((1, 2))
u9, t9 = cube['uts'][:, :, :3].mean((1, 2)), cube['toughness'][:, :, :3].mean((1, 2))
Q12, Q9 = quadrants(u12, t12), quadrants(u9, t9)
sizes = [int((Q12 == q).sum()) for q in (1, 2, 3, 4)]
Qs = [quadrants(cube['uts'][:, :, s].mean(1), cube['toughness'][:, :, s].mean(1)) for s in range(4)]
pair = [(Qs[a] != Qs[b]).mean() for a, b in itertools.combinations(range(4), 2)]
vs12 = [(Qs[s] != Q12).mean() for s in range(4)]
P(f"quadrant sizes (12 pulls): {sizes}")
P(f"two independent seeds (3 axes each) disagree on {100*np.mean(pair):.1f}% of networks (6 pairs, {100*min(pair):.1f}-{100*max(pair):.1f}%)")
P(f"one seed (3 axes) vs the 12-pull reference: {100*np.mean(vs12):.1f}% differ ({100*min(vs12):.1f}-{100*max(vs12):.1f}%); seed 1 (published basis): {100*vs12[0]:.1f}% ({int((Qs[0] != Q12).sum())})")
P(f"9-pull (seeds 1-3) vs 12-pull: {100*(Q9 != Q12).mean():.1f}% ({int((Q9 != Q12).sum())})")
P(f"Pearson r(UTS, toughness) 12-pull means: {np.corrcoef(u12, t12)[0, 1]:.3f}")
open(os.path.join(LOG_DIR, 'variance_output.txt'), 'w', encoding='utf-8').write("\n".join(out) + "\n")
print("\n".join(out))

# ---------------- LaTeX table
f = lambda x: f"{100*x:.1f}"
L = [r"% source: data/derived/mechanics/mechanics_pulls_4seeds.csv (327 x 3 axes x 4 seeds = 3,924 pulls)",
     r"% generated by generate_checks.ipynb (decomposition code verbatim from four_seed_analysis.py)",
     r"\begin{tabular}{lrr}", r"\toprule", r" & UTS & Toughness \\", r"\midrule",
     r"\multicolumn{3}{l}{\emph{Variance components (\% of total, 4 seeds)}} \\"]
for c, name in (('v_net', 'Between networks'), ('v_ax', 'Axis (systematic, $x/y/z$)'), ('v_int', 'Network $\\times$ axis'), ('v_err', 'Seed (pull-to-pull residual)')):
    L.append(f"\\quad {name} & {f(res['uts']['fr'][c])} & {f(res['toughness']['fr'][c])} \\\\")
L.append(f"\\quad $F$ (network $\\times$ axis vs seed) & {res['uts']['d4']['F_int']:.2f} & {res['toughness']['d4']['F_int']:.2f} \\\\")
L += [r"\midrule", r"\multicolumn{3}{l}{\emph{Reliability of the network mean, axes as a random facet}} \\"]
for key, name in (('rel1', '1 pull'), ('rel3', '3 axes $\\times$ 1 seed (published basis)'), ('rel9', '3 axes $\\times$ 3 seeds (9 pulls)'),
                  ('rel12', '3 axes $\\times$ 4 seeds (12 pulls)'), ('relinf', '3 axes $\\times$ unlimited seeds (ceiling)')):
    L.append(f"\\quad {name} & {res['uts'][key]:.2f} & {res['toughness'][key]:.2f} \\\\")
L += [r"\multicolumn{3}{l}{\emph{Reliability of the network mean, the three cube axes fixed}} \\"]
for key, name in (('relF3', '3 axes $\\times$ 1 seed'), ('relF12', '3 axes $\\times$ 4 seeds (12 pulls)')):
    L.append(f"\\quad {name} & {res['uts'][key]:.2f} & {res['toughness'][key]:.2f} \\\\")
L += [r"\midrule", r"\multicolumn{3}{l}{\emph{Quadrant stability (median split of UTS and toughness)}} \\",
      f"\\quad Two independent seeds (3 axes each) disagree & \\multicolumn{{2}}{{c}}{{{100*np.mean(pair):.1f}\\% (range {100*min(pair):.1f}--{100*max(pair):.1f}\\%, 6 pairs)}} \\\\",
      f"\\quad One seed vs the 12-pull reference & \\multicolumn{{2}}{{c}}{{{100*np.mean(vs12):.1f}\\% (range {100*min(vs12):.1f}--{100*max(vs12):.1f}\\%)}} \\\\",
      f"\\quad 9-pull vs 12-pull reference & \\multicolumn{{2}}{{c}}{{{100*(Q9 != Q12).mean():.1f}\\% ({int((Q9 != Q12).sum())} of 327)}} \\\\",
      f"\\quad Quadrant sizes Q1/Q2/Q3/Q4 (12 pulls) & \\multicolumn{{2}}{{c}}{{{sizes[0]}/{sizes[1]}/{sizes[2]}/{sizes[3]}}} \\\\",
      r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_variance.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")

# ---------------- figure (neutral style)
matplotlib.rcdefaults()   # start from matplotlib defaults, as the standalone script did
plt.rcParams.update({'text.color': '#0b0b0b', 'axes.labelcolor': '#0b0b0b', 'xtick.color': '#52514e', 'ytick.color': '#52514e', 'axes.edgecolor': '#c3c2b7', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False, 'font.family': 'DejaVu Sans'})
fig, (a, b) = plt.subplots(1, 2, figsize=(7.0, 2.6), gridspec_kw=dict(width_ratios=[1.15, 1]), constrained_layout=True)
comps = [('v_net', 'Between networks', '#2a78d6'), ('v_int', 'Network × axis', '#eb6834'), ('v_err', 'Seed (pull-to-pull)', '#1baf7a'), ('v_ax', 'Axis', '#eda100')]  # validated categorical slots 1-4
ys = [1, 0]; labs = ['UTS', 'Toughness']
for y, p in zip(ys, ('uts', 'toughness')):
    left = 0
    for c, name, col in comps:
        w = 100 * res[p]['fr'][c]
        a.barh(y, w, left=left, color=col, edgecolor='#fcfcfb', linewidth=1.5, height=0.55, label=name if p == 'uts' else None)
        if w > 8:
            a.text(left + w / 2, y, f"{w:.0f}%", ha='center', va='center', fontsize=7, color='#0b0b0b')
        left += w
a.set_yticks(ys); a.set_yticklabels(labs); a.set_xlim(0, 100); a.set_xlabel('Share of variance across 3,924 pulls (%)')
a.legend(frameon=False, fontsize=6.5, ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.32))
a.set_title('a  Variance components', loc='left', fontsize=8)
ms = np.arange(1, 9)
for p, col, lab in (('uts', '#2a78d6', 'UTS'), ('toughness', '#eb6834', 'Toughness')):
    d = res[p]['d4']
    b.plot(ms, [rel(d, 3, m) for m in ms], '-', color=col, lw=1.3, label=f'{lab}, axes random')
    b.plot(ms, [rel_fixed(d, 3, m) for m in ms], '--', color=col, lw=1.3, label=f'{lab}, axes fixed')
    b.plot([4], [rel(d, 3, 4)], 'o', color=col, ms=4); b.plot([4], [rel_fixed(d, 3, 4)], 'o', color=col, ms=4, mfc='white')
b.axvline(4, color='#999999', lw=0.6, ls=':')
b.set_xlabel('Velocity seeds per axis (3 axes each)'); b.set_ylabel('Reliability of network mean'); b.set_ylim(0, 1.0); b.set_xticks(ms)
b.legend(frameon=False, fontsize=6.5, ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.32))
b.set_title('b  Reliability of the network mean', loc='left', fontsize=8)
for ext in ('pdf', 'png'):
    fig.savefig(os.path.join(FIG_DIR, f'fig_variance_decomposition.{ext}'), dpi=300)
plt.close(fig)
print('wrote table_variance.tex and fig_variance_decomposition')

In [ ]:
# --- Statistics behind manuscript Figs. 6b and 7 (tables_checks/figure6_*.csv, figure5b_degree_bands_k12.csv) ---
# Graph descriptors from data/derived/mechanics/mechanics_12pull.csv (bitwise equal to dataset.pkl), pulls from
# mechanics_pulls_4seeds.csv; results are checked against the packaged tables in data/derived/figure_stats/.
from scipy.stats import ttest_ind as _ttest
cube, nets = load_pull_cube()
M = read_csv_exact(rev('mechanics', 'mechanics_12pull.csv')).set_index('folder_name').reindex(nets)
dc = [f'cnt_d{i}' for i in range(7)]
M['cycle_rank'] = sum(M[f'cnt_d{i}'] * i for i in range(7)) / 2.0 - (M[dc].sum(axis=1) - M['cnt_d0']) + 1   # active graph
assert np.allclose(M['cycle_rank'], M['cyclomatic_idx'])
M['deg_bimodality'] = M.apply(lambda r: ((skew(sum([[int(c.split('_d')[1])] * int(r[c]) for c in dc], [])) ** 2 + 1) /
    (kurtosis(sum([[int(c.split('_d')[1])] * int(r[c]) for c in dc], []), fisher=True) + 3)), axis=1)
FEATS = ['lambda_2', 'lambda_2_core', 'avg_path_len', 'cycle_rank', 'max_betweenness', 'num_bridges', 'deg_bimodality', 'deg_std', 'deg_entropy', 'avg_eigenvector_cent', 'assortativity']
NAMES = {'lambda_2': 'Algebraic connectivity (lambda_2)', 'lambda_2_core': 'Algebraic connectivity excl. dangling ends (lambda_2,core)', 'avg_path_len': 'Mean shortest path length (L)', 'cycle_rank': 'Cycle rank',
         'max_betweenness': 'Max. betweenness centrality', 'num_bridges': 'Bridges', 'deg_bimodality': 'Degree bimodality (D_bi)', 'deg_std': 'Degree heterogeneity (sigma_k)',
         'deg_entropy': 'Degree entropy (H)', 'avg_eigenvector_cent': 'Mean eigenvector centrality', 'assortativity': 'Assortativity (r)'}
TR = {'Toughening (weak, Q1->Q2)': (1, 2), 'Toughening (strong, Q3->Q4)': (3, 4), 'Strengthening (brittle, Q1->Q3)': (1, 3), 'Strengthening (tough, Q2->Q4)': (2, 4), 'Total (Q1->Q4)': (1, 4)}
X = M[FEATS].to_numpy(float); u = cube['uts'].mean((1, 2)); t = cube['toughness'].mean((1, 2)); n = len(nets)


def quad(u, t): return 1 + (t >= np.median(t)).astype(int) + 2 * (u >= np.median(u)).astype(int)


def cd(xa, xb):
    if len(xa) < 2 or len(xb) < 2: return np.nan
    sp = np.sqrt(((len(xa) - 1) * xa.var(ddof=1) + (len(xb) - 1) * xb.var(ddof=1)) / (len(xa) + len(xb) - 2)); return (xb.mean() - xa.mean()) / sp if sp else 0.0


Q = quad(u, t); sizes = {q: int((Q == q).sum()) for q in (1, 2, 3, 4)}
rng = np.random.default_rng(0); NB = 2000
boot_d = np.empty((NB, len(FEATS), len(TR))); boot_bars = np.empty((NB, len(FEATS), 2))
for b in range(NB):
    idx = rng.integers(0, n, n); Xb, qb = X[idx], quad(u[idx], t[idx])
    for j in range(len(FEATS)):
        ds = {k: cd(Xb[qb == a, j], Xb[qb == bq, j]) for k, (a, bq) in TR.items()}
        boot_d[b, j] = list(ds.values())
        boot_bars[b, j, 0] = abs(ds['Toughening (strong, Q3->Q4)']) - abs(ds['Toughening (weak, Q1->Q2)'])
        boot_bars[b, j, 1] = abs(ds['Strengthening (tough, Q2->Q4)']) - abs(ds['Strengthening (brittle, Q1->Q3)'])
rows = []
for j, f_ in enumerate(FEATS):
    for k, (name, (a, bq)) in enumerate(TR.items()):
        xa, xb = X[Q == a, j], X[Q == bq, j]; d = cd(xa, xb); p = _ttest(xa, xb, equal_var=False).pvalue
        lo, hi = np.nanpercentile(boot_d[:, j, k], [2.5, 97.5])
        rows.append(dict(metric=NAMES[f_], transition=name, n_from=len(xa), n_to=len(xb), cohens_d=d, ci_low=lo, ci_high=hi, welch_p=p))
S = pd.DataFrame(rows)
p = S.welch_p.to_numpy(); m = len(p); order = np.argsort(p); ranks = np.empty(m, int); ranks[order] = np.arange(1, m + 1)
q = p * m / ranks; q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]; bh_ = np.empty(m); bh_[order] = np.minimum(q_sorted, 1)
holm = np.empty(m); hs = np.maximum.accumulate((p[order] * (m - np.arange(m)))); holm[order] = np.minimum(hs, 1)
S['bh_q'] = bh_; S['holm_p'] = holm; S['sig_raw'] = S.welch_p < .05; S['sig_bh'] = S.bh_q < .05; S['sig_holm'] = S.holm_p < .05
S['ci_excludes_0'] = (S.ci_low > 0) | (S.ci_high < 0)
S.to_csv(os.path.join(TAB_DIR, 'figure6_stats.csv'), index=False)


def resid(y, Z): Z = np.column_stack([np.ones(len(y)), Z]); return y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]


cont = []
for j, f_ in enumerate(FEATS):
    x = X[:, j]; rT, pT = stats.pearsonr(x, t); rU, pU = stats.pearsonr(x, u)
    rTU = stats.pearsonr(resid(x, u), resid(t, u)); rUT = stats.pearsonr(resid(x, t), resid(u, t))
    cont.append(dict(metric=NAMES[f_], r_toughness=rT, p=pT, r_uts=rU, p_uts=pU, partial_r_T_given_U=rTU[0], p_T_given_U=rTU[1], partial_r_U_given_T=rUT[0], p_U_given_T=rUT[1]))
C = pd.DataFrame(cont); C.to_csv(os.path.join(TAB_DIR, 'figure6_continuous.csv'), index=False)
bars = []
for j, f_ in enumerate(FEATS):
    for col, lab in ((0, 'panel b: |d(Q3->Q4)| - |d(Q1->Q2)|'), (1, 'panel c: |d(Q2->Q4)| - |d(Q1->Q3)|')):
        pt = S[(S.metric == NAMES[f_])].set_index('transition').cohens_d
        val = (abs(pt['Toughening (strong, Q3->Q4)']) - abs(pt['Toughening (weak, Q1->Q2)'])) if col == 0 else (abs(pt['Strengthening (tough, Q2->Q4)']) - abs(pt['Strengthening (brittle, Q1->Q3)']))
        lo, hi = np.nanpercentile(boot_bars[:, j, col], [2.5, 97.5]); bars.append(dict(metric=NAMES[f_], bar=lab, delta_abs_d=val, ci_low=lo, ci_high=hi, ci_excludes_0=(lo > 0) or (hi < 0)))
B = pd.DataFrame(bars); B.to_csv(os.path.join(TAB_DIR, 'figure6_bars.csv'), index=False)
print(f"quadrant sizes (12-pull medians): Q1={sizes[1]} Q2={sizes[2]} Q3={sizes[3]} Q4={sizes[4]}")
print(f"55 cells: raw p<.05: {int(S.sig_raw.sum())}   BH q<.05: {int(S.sig_bh.sum())}   Holm p<.05: {int(S.sig_holm.sum())}   bootstrap CI excludes 0: {int(S.ci_excludes_0.sum())}")
print("panel b/c bars with CI excluding 0:", int(B.ci_excludes_0.sum()), 'of', len(B))

# degree bands of Fig. 6b (fig5b_bands_k12.py): 95% network bootstrap within each quadrant, seed 0, 2000 resamples
cnt = M[[f'cnt_d{i}' for i in range(7)]].to_numpy(float)
names = {1: 'Q1: Weak/Brittle', 2: 'Q2: Weak/Tough', 3: 'Q3: Strong/Brittle', 4: 'Q4: Strong/Tough'}


def pooled_pf(c):   # exactly as the notebook: pooled degree counts over the quadrant's networks
    tot = c.sum(0); return tot / tot.sum()


rng = np.random.default_rng(0); bands = {}
for q_ in (1, 2, 3, 4):
    c = cnt[Q == q_]; pf = pooled_pf(c); Bq = np.array([pooled_pf(c[rng.integers(0, len(c), len(c))]) for _ in range(2000)])
    bands[q_] = (pf, np.percentile(Bq, 2.5, axis=0), np.percentile(Bq, 97.5, axis=0), len(c))
rows = [dict(quadrant=names[q_], n=bands[q_][3], degree=k, p=bands[q_][0][k], ci_low=bands[q_][1][k], ci_high=bands[q_][2][k])
        for q_ in (1, 2, 3, 4) for k in range(7)]
pd.DataFrame(rows).to_csv(os.path.join(TAB_DIR, 'figure5b_degree_bands_k12.csv'), index=False)

# check against the packaged tables (the values behind the manuscript figures)
for fname in ('figure6_stats.csv', 'figure6_bars.csv', 'figure6_continuous.csv', 'figure5b_degree_bands_k12.csv'):
    check_same_table(read_csv_exact(os.path.join(TAB_DIR, fname)), read_csv_exact(rev('figure_stats', fname)), fname)

In [ ]:
# --- Cell, correlation and bar tables: statistics of manuscript Fig. 7 (and the nominal-stress column) ---
# Reads the statistics recomputed in the previous cell (tables_checks/) and data/derived/.
S = read_csv_exact(os.path.join(TAB_DIR, 'figure6_stats.csv'))
C = read_csv_exact(os.path.join(TAB_DIR, 'figure6_continuous.csv'))
B = read_csv_exact(os.path.join(TAB_DIR, 'figure6_bars.csv'))
BANDS = read_csv_exact(os.path.join(TAB_DIR, 'figure5b_degree_bands_k12.csv'))
M = read_csv_exact(rev('mechanics', 'mechanics_12pull.csv')).set_index('folder_name')
N = read_csv_exact(rev('stress_definition', 'network_true_vs_nominal.csv')).set_index('folder_name').reindex(M.index)
OLD = read_csv_exact(rev('stress_definition', 'fig6_cells_summary_12pull.csv'))
cube, _nets = load_pull_cube()
out = []; P = out.append

FEATS = ['lambda_2', 'lambda_2_core', 'avg_path_len', 'cycle_rank', 'max_betweenness', 'num_bridges', 'deg_bimodality', 'deg_std',
         'deg_entropy', 'avg_eigenvector_cent', 'assortativity']
NAMES = {'lambda_2': 'Algebraic connectivity (lambda_2)', 'lambda_2_core': 'Algebraic connectivity excl. dangling ends (lambda_2,core)',
         'avg_path_len': 'Mean shortest path length (L)', 'cycle_rank': 'Cycle rank', 'max_betweenness': 'Max. betweenness centrality',
         'num_bridges': 'Bridges', 'deg_bimodality': 'Degree bimodality (D_bi)', 'deg_std': 'Degree heterogeneity (sigma_k)',
         'deg_entropy': 'Degree entropy (H)', 'avg_eigenvector_cent': 'Mean eigenvector centrality', 'assortativity': 'Assortativity (r)'}
TEX = {'lambda_2': r'Algebraic connectivity ($\lambda_2$)', 'lambda_2_core': r'Alg.\ conn.\ excl.\ dangling ends ($\lambda_{2,\mathrm{core}}$)',
       'avg_path_len': r'Mean shortest path length ($L$)', 'cycle_rank': 'Cycle rank', 'max_betweenness': 'Max.\\ betweenness centrality',
       'num_bridges': 'Bridges', 'deg_bimodality': r'Degree bimodality ($D_{bi}$)', 'deg_std': r'Degree heterogeneity ($\sigma_k$)',
       'deg_entropy': r'Degree entropy ($H$)', 'avg_eigenvector_cent': 'Mean eigenvector centrality', 'assortativity': r'Assortativity ($r$)'}
TR = {'Toughening (weak, Q1->Q2)': (1, 2), 'Toughening (strong, Q3->Q4)': (3, 4), 'Strengthening (brittle, Q1->Q3)': (1, 3),
      'Strengthening (tough, Q2->Q4)': (2, 4), 'Total (Q1->Q4)': (1, 4)}
TRTEX = {'Toughening (weak, Q1->Q2)': r'Q1$\to$Q2', 'Toughening (strong, Q3->Q4)': r'Q3$\to$Q4', 'Strengthening (brittle, Q1->Q3)': r'Q1$\to$Q3',
         'Strengthening (tough, Q2->Q4)': r'Q2$\to$Q4', 'Total (Q1->Q4)': r'Q1$\to$Q4'}
X = M.copy(); X['cycle_rank'] = X['cyclomatic_idx'].astype(float)   # active graph, E - N_active + 1


def quad(u, t): return 1 + (t >= np.median(t)).astype(int) + 2 * (u >= np.median(u)).astype(int)


def cd(xa, xb):
    sp = np.sqrt(((len(xa) - 1) * xa.var(ddof=1) + (len(xb) - 1) * xb.var(ddof=1)) / (len(xa) + len(xb) - 2)); return (xb.mean() - xa.mean()) / sp


def bh(p):
    p = np.asarray(p); m = len(p); order = np.argsort(p); ranks = np.empty(m, int); ranks[order] = np.arange(1, m + 1)
    q = p * m / ranks; qs = np.minimum.accumulate(q[order][::-1])[::-1]; o = np.empty(m); o[order] = np.minimum(qs, 1); return o


def cells(u, t):
    Q = quad(u, t); rows = []
    for f in FEATS:
        x = X[f].to_numpy(float)
        for name, (a, b) in TR.items():
            xa, xb = x[Q == a], x[Q == b]
            rows.append(dict(key=f, metric=NAMES[f], transition=name, n_from=len(xa), n_to=len(xb), d=cd(xa, xb), p=ttest_ind(xa, xb, equal_var=False).pvalue))
    R = pd.DataFrame(rows); R['q'] = bh(R.p); return R, Q


# 1. true stress, reproduce
assert np.allclose(N.UTS_true_12pull, M.uts_mean) and np.allclose(N.tough_true_12pull, M.toughness_mean)
T, Qt = cells(M.uts_mean.to_numpy(), M.toughness_mean.to_numpy())
assert (T.metric.values == S.metric.values).all() and (T.transition.values == S.transition.values).all()
P(f"true stress reproduction: max|d diff| {np.abs(T.d - S.cohens_d).max():.2e}  max|p diff| {np.abs(T.p - S.welch_p).max():.2e}  max|q diff| {np.abs(T.q - S.bh_q).max():.2e}")
P(f"true: quadrant sizes {[int((Qt == q).sum()) for q in (1, 2, 3, 4)]}; raw {int((S.welch_p < .05).sum())}, BH {int(S.sig_bh.sum())}, Holm {int(S.sig_holm.sum())}, CI excl 0 {int(S.ci_excludes_0.sum())}")
# 2. nominal stress
Nm, Qn = cells(N.UTS_nom_12pull.to_numpy(), N.tough_nom_12pull.to_numpy())
P(f"nominal: quadrant sizes {[int((Qn == q).sum()) for q in (1, 2, 3, 4)]}; networks changing quadrant {int((Qn != Qt).sum())}; BH-significant {int((Nm.q < .05).sum())}")
S['sig_nom'] = (Nm.q < .05).values; S['d_nom'] = Nm.d.values; S['q_nom'] = Nm.q.values; S['key'] = T.key.values
both = int((S.sig_bh & S.sig_nom).sum())
P(f"cells BH-significant under BOTH true and nominal stress (corrected cycle rank): {both}")
P(f"true only: {int((S.sig_bh & ~S.sig_nom).sum())}; nominal only: {int((~S.sig_bh & S.sig_nom).sum())}; neither: {int((~S.sig_bh & ~S.sig_nom).sum())}")
# 3. cross-check against the older-convention file
OLDk = OLD.set_index(['metric', 'transition'])
trshort = {k: f"Q{a}->Q{b}" for k, (a, b) in TR.items()}
mism = []
for _, r in S.iterrows():
    o = OLDk.loc[(r.key, trshort[r.transition])]
    if bool(o.sig_nom) != bool(r.sig_nom):
        mism.append((r.key, trshort[r.transition], bool(o.sig_nom), bool(r.sig_nom)))
non_cyc = [m for m in mism if m[0] != 'cycle_rank']
P(f"flag mismatches vs fig6_cells_summary_12pull.csv sig_nom: non-cycle-rank {len(non_cyc)} {non_cyc}; cycle-rank {[m for m in mism if m[0] == 'cycle_rank']}")
P(f"older-convention file: both-significant count {int((OLD.sig_true & OLD.sig_nom).sum())}")
cyc = S[S.key == 'cycle_rank'][['transition', 'cohens_d', 'bh_q', 'd_nom', 'q_nom', 'sig_nom']]
P("cycle rank (corrected) true vs nominal:\n" + cyc.to_string(index=False))
lost = S[S.sig_bh & ~S.sig_nom][['key', 'transition', 'cohens_d', 'bh_q', 'd_nom', 'q_nom']]
gain = S[~S.sig_bh & S.sig_nom][['key', 'transition', 'cohens_d', 'bh_q', 'd_nom', 'q_nom']]
P("true-only cells:\n" + lost.to_string(index=False)); P("nominal-only cells:\n" + gain.to_string(index=False))
S.drop(columns=['key']).to_csv(os.path.join(LOG_DIR, 'table_cells_with_nominal.csv'), index=False)

# 4. degree bands (Fig. 6b), reproducing fig5b_bands_k12.py
nets = [str(s) for s in cube['nets']]
cnt = M.reindex(nets)[[f'cnt_d{i}' for i in range(7)]].to_numpy(float)
u, t = cube['uts'].mean((1, 2)), cube['toughness'].mean((1, 2))
assert np.allclose(u, M.reindex(nets).uts_mean) and np.allclose(t, M.reindex(nets).toughness_mean)
Qb = 1 + (t >= np.median(t)).astype(int) + 2 * (u >= np.median(u)).astype(int)
pooled = lambda c: c.sum(0) / c.sum()
rng = np.random.default_rng(0); boot = {}; pf = {}
for q in (1, 2, 3, 4):
    c = cnt[Qb == q]; pf[q] = pooled(c); boot[q] = np.array([pooled(c[rng.integers(0, len(c), len(c))]) for _ in range(2000)])
chk = max(abs(BANDS[(BANDS.quadrant.str.startswith(f'Q{q}')) & (BANDS.degree == k)].p.iloc[0] - pf[q][k]) for q in (1, 2, 3, 4) for k in range(7))
chk_ci = max(abs(BANDS[(BANDS.quadrant.str.startswith(f'Q{q}')) & (BANDS.degree == k)].ci_low.iloc[0] - np.percentile(boot[q][:, k], 2.5)) for q in (1, 2, 3, 4) for k in range(7))
P(f"band reproduction vs figure5b_degree_bands_k12.csv: max|dp| {chk:.1e}, max|d ci_low| {chk_ci:.1e}")
bands_txt = {}
for (a, b) in ((1, 2), (3, 4)):
    for k in (1, 2, 6):
        diff = boot[b][:, k] - boot[a][:, k]; pt = pf[b][k] - pf[a][k]; lo, hi = np.percentile(diff, [2.5, 97.5]); p2 = 2 * min((diff > 0).mean(), (diff < 0).mean())
        bands_txt[(a, b, k)] = (100 * pt, 100 * lo, 100 * hi, p2)
        P(f"  P(f={k}) Q{a}->Q{b}: {100*pt:+.2f} pp  95% CI [{100*lo:+.2f}, {100*hi:+.2f}]  bootstrap two-sided p ~ {p2:.3f}")
open(os.path.join(LOG_DIR, 'stats_output.txt'), 'w', encoding='utf-8').write("\n".join(out) + "\n")
print("\n".join(out))


# ---------- LaTeX
def fp(p):
    if p < 1e-3:
        m, e = f"{p:.1e}".split('e'); return f"${m}\\times10^{{{int(e)}}}$"
    return f"{p:.3f}"


yn = lambda b: r"yes" if b else r"no"
L = [r"% source: tables_checks/figure6_stats.csv (true stress; d, CI, Welch p, BH q, Holm p, n)",
     r"% nominal column: computed in generate_checks.ipynb from data/derived/stress_definition/network_true_vs_nominal.csv",
     r"\begingroup\footnotesize\setlength{\tabcolsep}{4pt}",
     r"\begin{longtable}{lcrcccccc}",
     r"\caption{Statistics of the 55 cells of the Cohen's $d$ heatmap (Fig.~7a): 11 graph metrics $\times$ 5 quadrant transitions, quadrants from the median split of the 12-pull network means of peak true stress (UTS) and toughness. $d$ is (mean of the destination quadrant $-$ mean of the origin quadrant) divided by the pooled standard deviation. CI: 95\% percentile interval over 2000 network bootstrap resamples with the quadrants re-derived in every resample. $p$: Welch's two-sample $t$ test. $q$: Benjamini--Hochberg over the 55 tests. Holm: Holm-adjusted $p$. Last column: whether the cell is also Benjamini--Hochberg significant ($q<0.05$) when the quadrants are defined with nominal stress; cycle rank is $E-N_\mathrm{active}+1$ in both columns.}\label{tab:cells}\\",
     r"\toprule", r"Transition & $n_\mathrm{from}/n_\mathrm{to}$ & $d$ & 95\% CI of $d$ & Welch $p$ & BH $q$ & Holm $p$ & BH sig. & Nominal sig. \\", r"\midrule", r"\endfirsthead",
     r"\toprule", r"Transition & $n_\mathrm{from}/n_\mathrm{to}$ & $d$ & 95\% CI of $d$ & Welch $p$ & BH $q$ & Holm $p$ & BH sig. & Nominal sig. \\", r"\midrule", r"\endhead",
     r"\bottomrule", r"\endfoot"]
prev = None
for _, r in S.iterrows():
    if r.key != prev:
        if prev is not None: L.append(r"\addlinespace[3pt]")
        L.append(r"\multicolumn{9}{l}{\textbf{" + TEX[r.key] + r"}} \\*")   # group header; no page break right after it
    prev = r.key
    L.append(f"\\quad {TRTEX[r.transition]} & {r.n_from}/{r.n_to} & ${r.cohens_d:+.2f}$ & $[{r.ci_low:+.2f}, {r.ci_high:+.2f}]$ & {fp(r.welch_p)} & {fp(r.bh_q)} & {fp(r.holm_p)} & {yn(r.sig_bh)} & {yn(r.sig_nom)} \\\\")
L += [r"\end{longtable}", r"\endgroup"]
open(os.path.join(TAB_DIR, 'table_cells.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")

key_of = {v: k for k, v in NAMES.items()}
L = [r"% source: tables_checks/figure6_continuous.csv", r"\begin{tabular}{lrrrrrrrr}", r"\toprule",
     r"Metric & $r$(toughness) & $p$ & $r$(UTS) & $p$ & $r$(T$\,|\,$UTS) & $p$ & $r$(UTS$\,|\,$T) & $p$ \\", r"\midrule"]
for _, r in C.iterrows():
    L.append(f"{TEX[key_of[r.metric]]} & ${r.r_toughness:+.3f}$ & {fp(r.p)} & ${r.r_uts:+.3f}$ & {fp(r.p_uts)} & ${r.partial_r_T_given_U:+.3f}$ & {fp(r.p_T_given_U)} & ${r.partial_r_U_given_T:+.3f}$ & {fp(r.p_U_given_T)} \\\\")
L += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_continuous.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")

L = [r"% source: tables_checks/figure6_bars.csv", r"\begin{tabular}{lrcrc}", r"\toprule",
     r" & \multicolumn{2}{c}{Panel b: $|d_{\mathrm{Q3\to Q4}}|-|d_{\mathrm{Q1\to Q2}}|$} & \multicolumn{2}{c}{Panel c: $|d_{\mathrm{Q2\to Q4}}|-|d_{\mathrm{Q1\to Q3}}|$} \\",
     r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}", r"Metric & $\Delta|d|$ & 95\% CI & $\Delta|d|$ & 95\% CI \\", r"\midrule"]
for f in FEATS:
    sb = B[(B.metric == NAMES[f]) & B.bar.str.startswith('panel b')].iloc[0]; sc = B[(B.metric == NAMES[f]) & B.bar.str.startswith('panel c')].iloc[0]
    L.append(f"{TEX[f]} & ${sb.delta_abs_d:+.2f}$ & $[{sb.ci_low:+.2f}, {sb.ci_high:+.2f}]$ & ${sc.delta_abs_d:+.2f}$ & $[{sc.ci_low:+.2f}, {sc.ci_high:+.2f}]$ \\\\")
L += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_bars.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")
print("bars with CI excluding 0:", int(B.ci_excludes_0.sum()), "of", len(B))

In [ ]:
# --- Atomistic Tg tables (O-O check, Tg per cooling history, lag scan) ---
# The MSD per temperature, lag and cooling history is read from data/derived/tg/tg_msd_histories.csv
# (the estimator of make_figure5_replicates.py applied to the msd.dat files); the O-O table from oo_bond_check.csv.
TGD = read_csv_exact(rev('tg', 'tg_msd_histories.csv'))
CHEMS = {"PDMS": ("PDMS", 150), "PMTFPS": ("FPDMS", 200)}
DPS = ["DP_10", "DP_30", "DP_100"]
HISTS = ("orig", "rep2", "rep3")
HNAME = {"orig": "original", "rep2": "replicate2", "rep3": "replicate3"}
LAGS = (2, 3, 4, 5, 6, 8)


# --- verbatim from make_figure5_replicates.py --------------------------------
def piecewise_linear(T, Tg, m1, m2, c1):
    condlist = [T < Tg, T >= Tg]
    funclist = [lambda x: m1 * x + c1,
                lambda x: m2 * (x - Tg) + (m1 * Tg + c1)]
    return np.piecewise(T, condlist, funclist)


def get_optimized_bilinear_tg(temp, msd, guess_tg=150):
    try:
        p0 = [guess_tg, 0.005, 0.02, np.min(msd)]
        bounds = ([np.min(temp) + 5, -np.inf, -np.inf, -np.inf],
                  [np.max(temp) - 5, np.inf, np.inf, np.inf])
        popt, pcov = curve_fit(piecewise_linear, temp, msd, p0=p0,
                               bounds=bounds, maxfev=10000)
        return popt[0], (popt, pcov)
    except Exception:
        return np.nan, None


def load_lag(label, dp, hist, lag):
    """{T: MSD at `lag` ps} from the tidy table (the original script parsed msd.dat)."""
    s = TGD[(TGD.system == label) & (TGD.dp == int(dp[3:])) & (TGD.history == HNAME[hist])]
    if len(s) == 0:
        raise ValueError(f"no MSD data for {label} {dp} {hist}")
    return dict(zip(s.Temp.astype(int).tolist(), s[f"MSD_{lag}ps"].to_numpy()))


# --- verbatim from analyze_lag_sensitivity.py (grid-descent notebook convention) ---------
def fit_local(T, y, start):
    def sse(t):
        X = np.column_stack([np.ones_like(T), T, np.maximum(T - t, 0)])
        c, *_ = np.linalg.lstsq(X, y, rcond=None)
        return float(np.sum((y - X @ c) ** 2))
    grid = np.linspace(T[3], T[-4], 1001)
    s = np.array([sse(t) for t in grid])
    i = int(np.argmin(np.abs(grid - start)))
    while 0 < i < len(s) - 1:
        if s[i - 1] < s[i]: i -= 1
        elif s[i + 1] < s[i]: i += 1
        else: break
    X = np.column_stack([np.ones_like(T), T])
    c2, *_ = np.linalg.lstsq(X, y, rcond=None)
    lin = float(np.sum((y - X @ c2) ** 2))
    return grid[i], 100 * (1 - s[i] / lin)


res = {}
for lag in LAGS:
    for label, (folder, guess) in CHEMS.items():
        for dp in DPS:
            per = [load_lag(label, dp, h, lag) for h in HISTS]
            Ts = np.array(sorted(set.intersection(*[set(p) for p in per])), float)
            M = np.array([[p[t] for t in Ts] for p in per])
            tg_i = [float(get_optimized_bilinear_tg(Ts, m, guess)[0]) for m in M]
            tg_grid = [fit_local(Ts, m, float(guess)) for m in M]
            tg_avg = float(get_optimized_bilinear_tg(Ts, M.mean(0), guess)[0])
            res[(lag, label, dp)] = dict(nT=len(Ts), Tmin=Ts.min(), Tmax=Ts.max(), tg_i=tg_i, tg_avg=tg_avg,
                                         tg_sd=float(np.std(tg_i, ddof=1)),
                                         tg_grid=[g[0] for g in tg_grid], fitq_grid=[g[1] for g in tg_grid])

lines = []
P = lines.append
P("Tg per history: 4 ps, manuscript curve_fit routine (make_figure5_replicates.py)")
for label in CHEMS:
    for dp in DPS:
        r = res[(4, label, dp)]
        P(f"{label:7s}{dp:7s} nT={r['nT']} T={r['Tmin']:.0f}-{r['Tmax']:.0f}  per-history " +
          " ".join(f"{v:6.1f}" for v in r['tg_i']) +
          f" | mean {np.mean(r['tg_i']):6.1f} spread {max(r['tg_i'])-min(r['tg_i']):5.1f} SD {r['tg_sd']:4.1f} | avg-curve {r['tg_avg']:6.1f}"
          f" || grid convention " + " ".join(f"{v:6.1f}" for v in r['tg_grid']) +
          " fitq% " + " ".join(f"{v:4.0f}" for v in r['fitq_grid']))
P("\ncontrast PMTFPS - PDMS, 4 ps (curve_fit per history | avg-curve) || grid convention")
for dp in DPS:
    a, b = res[(4, 'PDMS', dp)], res[(4, 'PMTFPS', dp)]
    c = [y - x for x, y in zip(a['tg_i'], b['tg_i'])]
    cg = [y - x for x, y in zip(a['tg_grid'], b['tg_grid'])]
    P(f"{dp:7s} " + " ".join(f"{v:6.1f}" for v in c) + f" | mean {np.mean(c):5.1f} spread {max(c)-min(c):5.1f} SD {np.std(c, ddof=1):4.1f}"
      f" | avg-curve {b['tg_avg']-a['tg_avg']:5.1f} || grid " + " ".join(f"{v:6.1f}" for v in cg) +
      f" mean {np.mean(cg):5.1f} spread {max(cg)-min(cg):5.1f}")
P("\nlag scan: Tg of the history-averaged curve at DP 100 (curve_fit)")
for lag in LAGS:
    a, b = res[(lag, 'PDMS', 'DP_100')], res[(lag, 'PMTFPS', 'DP_100')]
    P(f"lag {lag} ps: PMTFPS {b['tg_avg']:6.1f}  PDMS {a['tg_avg']:6.1f}  contrast {b['tg_avg']-a['tg_avg']:5.1f}"
      f"  | per-history contrast mean {np.mean([y-x for x, y in zip(a['tg_i'], b['tg_i'])]):5.1f}")
txt = "\n".join(lines)
print(txt)
open(os.path.join(LOG_DIR, "tg_tables_output.txt"), "w", encoding='utf-8').write(txt + "\n")
json.dump({f"{k[0]}|{k[1]}|{k[2]}": v for k, v in res.items()}, open(os.path.join(LOG_DIR, "tg_tables.json"), "w"), indent=1,
          default=float)

# ---------------- O-O bond check (cells verbatim from the replicate manifest)
oo = pd.read_csv(rev('tg', 'oo_bond_check.csv'), dtype=str)
rows = oo[['system', 'atoms', 'bonds', 'OO_bonds_system_data', 'OO_bonds_ready2deform_data']].values.tolist()
L = [r"% source: data/derived/tg/oo_bond_check.csv (O-O (peroxide) bond check of the replicate simulations)",
     r"\begin{tabular}{lrrrr}", r"\toprule",
     r" & & & \multicolumn{2}{c}{O--O bonds} \\", r"\cmidrule(lr){4-5}",
     r"System & Atoms & Bonds & \texttt{system.data} & \texttt{ready2deform.data} \\", r"\midrule"]
for r in rows:
    L.append(" & ".join([r[0].replace('FPDMS', 'PMTFPS')] + r[1:]) + r" \\")
L += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_OO.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")

# ---------------- Tg per history, lag scan
J = json.load(open(os.path.join(LOG_DIR, 'tg_tables.json')))
g = lambda lag, chem, dp: J[f"{lag}|{chem}|{dp}"]
L = [r"% source: data/derived/tg/tg_msd_histories.csv (msd.dat of the three cooling histories, data/raw/tg_cooling.tar.xz),",
     r"%         fitted in generate_checks.ipynb -> tables_checks/out/tg_tables.json",
     r"% curve_fit columns reproduce the per-history values behind Fig. 5; grid columns use the grid-search convention",
     r"\begin{tabular}{llrrrrrrcrrr}", r"\toprule",
     r" & & \multicolumn{6}{c}{Two-slope \texttt{curve\_fit} (as in Fig.~5)} & & \multicolumn{3}{c}{Grid search, same start} \\",
     r"\cmidrule(lr){3-8}\cmidrule(lr){10-12}",
     r"System & DP & Orig. & Rep.~2 & Rep.~3 & Mean & Spread & Avg.\ curve & & Orig. & Rep.~2 & Rep.~3 \\", r"\midrule"]
for chem in ('PDMS', 'PMTFPS'):
    for dp in DPS:
        r = g(4, chem, dp); t = r['tg_i']; gr = r['tg_grid']
        L.append(f"{chem if dp == 'DP_10' else ''} & {dp[3:]} & " + " & ".join(f"{v:.1f}" for v in t) +
                 f" & {np.mean(t):.1f} & {max(t)-min(t):.1f} & {r['tg_avg']:.1f} & & " + " & ".join(f"{v:.1f}" for v in gr) + r" \\")
    L.append(r"\midrule")
for dp in DPS:
    a, b = g(4, 'PDMS', dp), g(4, 'PMTFPS', dp)
    c = [y - x for x, y in zip(a['tg_i'], b['tg_i'])]; cg = [y - x for x, y in zip(a['tg_grid'], b['tg_grid'])]
    L.append(f"{'Contrast' if dp == 'DP_10' else ''} & {dp[3:]} & " + " & ".join(f"{v:.1f}" for v in c) +
             f" & {np.mean(c):.1f} & {max(c)-min(c):.1f} & {b['tg_avg']-a['tg_avg']:.1f} & & " + " & ".join(f"{v:.1f}" for v in cg) + r" \\")
L += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_Tg.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")

L = [r"% source: tables_checks/out/tg_tables.json (manuscript curve_fit on the history-averaged MSD curve, DP 100)",
     r"\begin{tabular}{lrrrrrr}", r"\toprule", r"MSD lag (ps) & 2 & 3 & 4 & 5 & 6 & 8 \\", r"\midrule"]
L.append("PMTFPS $T_g$ (K) & " + " & ".join(f"{g(l, 'PMTFPS', 'DP_100')['tg_avg']:.1f}" for l in LAGS) + r" \\")
L.append("PDMS $T_g$ (K) & " + " & ".join(f"{g(l, 'PDMS', 'DP_100')['tg_avg']:.1f}" for l in LAGS) + r" \\")
L.append("Contrast (K) & " + " & ".join(f"{g(l, 'PMTFPS', 'DP_100')['tg_avg'] - g(l, 'PDMS', 'DP_100')['tg_avg']:.1f}" for l in LAGS) + r" \\")
L.append("SD over histories (K), PMTFPS/PDMS & " + " & ".join(f"{g(l, 'PMTFPS', 'DP_100')['tg_sd']:.1f}/{g(l, 'PDMS', 'DP_100')['tg_sd']:.1f}" for l in LAGS) + r" \\")
L += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_lag.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")
print('\nTg tables written')

In [ ]:
# --- Choice of the 4 ps MSD lag ---
# Plotting code of scripts/raw_data_analyses/tg/make_lag_rationale_figure.py. The history MSD curves at lags 0-10 ps
# come from data/derived/tg/tg_msd_histories.csv (same estimator, averaged over the 10 ps windows).
from npj_style_v1 import set_npj_style

HISTS = ("orig", "rep2", "rep3")
DP = "DP_100"
CHEM = {"PDMS": ("PDMS", 150, "#0173B2"), "PMTFPS": ("FPDMS", 200, "#DE8F05")}


def piecewise_linear(T, Tg, m1, m2, c1):
    return np.piecewise(T, [T < Tg, T >= Tg],
                        [lambda x: m1 * x + c1,
                         lambda x: m2 * (x - Tg) + (m1 * Tg + c1)])


def fit_tg(temp, msd, guess):
    try:
        p0 = [guess, 0.005, 0.02, np.min(msd)]
        bounds = ([np.min(temp) + 5, -np.inf, -np.inf, -np.inf],
                  [np.max(temp) - 5, np.inf, np.inf, np.inf])
        popt, _ = curve_fit(piecewise_linear, temp, msd, p0=p0, bounds=bounds, maxfev=10000)
        return popt[0]
    except Exception:
        return np.nan


def load_curves(label, hist):
    """{T: MSD at lags 0..10 ps}, averaged over the 10 ps windows, from the tidy table."""
    s = TGD[(TGD.system == label) & (TGD.dp == int(DP[3:])) & (TGD.history == HNAME[hist])]
    arr = s[[f"MSD_{l}ps" for l in range(11)]].to_numpy(float)
    return {int(T): arr[i] for i, T in enumerate(s.Temp)}


data = {}
for label, (folder, guess, col) in CHEM.items():
    per = [load_curves(label, h) for h in HISTS]
    Ts = sorted(set.intersection(*[set(p) for p in per]))
    data[label] = dict(T=np.array(Ts, float),
                       per=[np.array([p[t] for t in Ts]) for p in per])   # 3 x (nT, 11)
    data[label]["avg"] = np.mean(data[label]["per"], axis=0)

t = np.arange(11, dtype=float)                     # lags 0..10 ps
tm = np.sqrt(t[1:-1] * t[2:])                      # geometric mid-lags 1.4..9.5
LAGS9 = list(range(1, 10))

matplotlib.rcdefaults()   # start from matplotlib defaults, as the standalone script did
width, _ = set_npj_style(column_type='double')
fig, (a, b, c) = plt.subplots(1, 3, figsize=(width, width * 0.36), constrained_layout=True)

# ---- a: the cage, seen directly ---------------------------------------------
d = data["PDMS"]
show = [(110, "#08519C", "110 K"), (150, "#8C8C8C", "150 K ≈ Tg"),
        (220, "#E6550D", "220 K"), (300, "#A63603", "300 K")]
for T, col, lab in show:
    i = list(d["T"]).index(T)
    a.plot(t, d["avg"][i], color=col, lw=1.3, marker="o", ms=2.6)
    a.annotate(lab, (10, d["avg"][i][-1]), textcoords="offset points",
               xytext=(4, 0), color=col, fontsize=6.5, va="center")
a.axvspan(2.5, 4.5, color="#bdbdbd", alpha=0.25, lw=0)
a.axvline(4, color="#555555", lw=0.7, ls=(0, (3, 2)))
a.text(0.25, 2.2, "fast: the atom hits its\ncage within ~1 ps", fontsize=6.5,
       color="#333333", ha="left", va="top")
a.text(5.0, 1.03, "then only slow creep\ninside the cage", fontsize=6.5,
       color="#333333", ha="left", va="center")
a.set_xlim(0, 13.2)
a.set_ylim(0, 2.25)
a.set_xlabel("time (ps)")
a.set_ylabel(r"MSD ($\mathrm{\AA}^2$)")
a.set_title("PDMS, DP 100", fontsize=7.5, loc="left")

# ---- b: where the rattling regime is -----------------------------------------
for label, (folder, guess, col) in CHEM.items():
    d = data[label]
    for T, ls in ((110, (0, (3, 2))), (150 if label == "PDMS" else 190, "-")):
        i = list(d["T"]).index(T)
        m = d["avg"][i]
        alpha = np.diff(np.log(m[1:])) / np.diff(np.log(t[1:]))
        b.plot(tm, alpha, color=col, lw=1.2, ls=ls, marker="o", ms=2.4)
b.axvspan(2.5, 4.5, color="#bdbdbd", alpha=0.25, lw=0)
b.axvline(4, color="#555555", lw=0.7, ls=(0, (3, 2)))
b.text(3.5, 0.012, "plateau: slowest growth", ha="center", va="bottom",
       fontsize=6.5, color="#333333")
b.plot([], [], color="#0173B2", lw=1.2, label="PDMS")
b.plot([], [], color="#DE8F05", lw=1.2, label="PMTFPS")
b.plot([], [], color="#666666", lw=1.2, label="at its Tg")
b.plot([], [], color="#666666", lw=1.2, ls=(0, (3, 2)), label="deep glass, 110 K")
b.legend(frameon=False, fontsize=6, loc="upper right", ncol=1)
b.set_xlabel("time (ps)")
b.set_ylabel(r"growth rate  d ln MSD / d ln $t$")
b.set_xlim(1, 10)
b.set_ylim(0.0, 0.47)

# ---- c: Tg depends on the probe window, like every experimental Tg -----------
for label, (folder, guess, col) in CHEM.items():
    d = data[label]
    mean, sd = [], []
    for L_ in LAGS9:
        tg_h = [fit_tg(d["T"], p[:, L_], guess) for p in d["per"]]
        mean.append(fit_tg(d["T"], d["avg"][:, L_], guess))
        sd.append(np.nanstd(tg_h, ddof=1))
    mean, sd = np.array(mean), np.array(sd)
    c.fill_between(LAGS9, mean - sd, mean + sd, color=col, alpha=0.18, lw=0)
    c.plot(LAGS9, mean, color=col, lw=1.4, marker="o", ms=2.8)
    c.annotate(label, (LAGS9[-1], mean[-1]), textcoords="offset points",
               xytext=(4, 0), color=col, fontsize=6.5, va="center")
    print(label, "Tg vs lag:", " ".join(f"{L_}ps {m:.1f}+/-{s:.1f}"
                                        for L_, m, s in zip(LAGS9, mean, sd)))
c.axvspan(2.5, 4.5, color="#bdbdbd", alpha=0.25, lw=0)
c.axvline(4, color="#555555", lw=0.7, ls=(0, (3, 2)))
c.text(4.15, 206, "4 ps", fontsize=6.5, color="#333333")
c.set_xlabel("probe window (ps)")
c.set_ylabel(r"apparent $T_g$ (K)")
c.set_xlim(0.6, 10.6)
c.set_ylim(115, 212)

for ax_, lab in zip((a, b, c), "abc"):
    ax_.text(-0.02, 1.02, lab, transform=ax_.transAxes, fontsize=9,
             fontweight="bold", va="bottom", ha="right")

for ext in ("pdf", "png"):
    fig.savefig(os.path.join(FIG_DIR, f"fig_Tg_lag_rationale.{ext}"), dpi=600)
plt.close(fig)
print("lag figure written")

In [ ]:
# --- Chain statistics of the atomistic networks ---
# Reads the deposited data/csv/analysis_ree_samples.csv and analysis_rg_samples.csv.
ree = read_csv_exact(os.path.join(DATA_PATH, 'csv', 'analysis_ree_samples.csv'))
rg = read_csv_exact(os.path.join(DATA_PATH, 'csv', 'analysis_rg_samples.csv'))
out = []
P = out.append
P("systems: " + str(sorted(ree.system.unique())) + "  counts: " + str(ree.groupby(['system', 'dp']).size().to_dict()))
DPS12 = (10, 30, 100)
rng = np.random.default_rng(20260925); NB = 2000


def nu_of(vals_by_dp):
    y = np.log([np.mean(v ** 2) for v in vals_by_dp]); x = np.log(DPS12)
    return np.polyfit(x, y, 1)[0] / 2


rows = []
nus = {}
for chem in sorted(ree.system.unique(), key=lambda s: 0 if s == 'PDMS' else 1):
    R = [ree[(ree.system == chem) & (ree.dp == d)].sort_values('sample_idx').ree.to_numpy(float) for d in DPS12]
    G = [rg[(rg.system == chem) & (rg.dp == d)].sort_values('sample_idx').rg.to_numpy(float) for d in DPS12]
    nuR, nuG = nu_of(R), nu_of(G)
    bR, bG = [], []
    for _ in range(NB):
        idx = [rng.integers(0, len(v), len(v)) for v in R]
        bR.append(nu_of([v[i] for v, i in zip(R, idx)]))
        bG.append(nu_of([v[i] for v, i in zip(G, idx)]))    # same strands for Rg (paired resample)
    nus[chem] = (nuR, np.std(bR, ddof=1), nuG, np.std(bG, ddof=1))
    for d, r, g_ in zip(DPS12, R, G):
        assert len(r) == len(g_) == 210
        rows.append(dict(chem=chem, dp=d, n=len(r), R2=np.mean(r ** 2), G2=np.mean(g_ ** 2), ratio=np.mean(r ** 2) / np.mean(g_ ** 2)))
    P(f"{chem}: nu_Ree {nuR:.4f} +/- {np.std(bR, ddof=1):.4f}   nu_Rg {nuG:.4f} +/- {np.std(bG, ddof=1):.4f}")
T = pd.DataFrame(rows)
P(T.to_string(index=False))
open(os.path.join(LOG_DIR, 'chain_stats_output.txt'), 'w', encoding='utf-8').write("\n".join(out) + "\n")
print("\n".join(out))

label = {'PDMS': 'PDMS', 'FPDMS': 'PMTFPS', 'PMTFPS': 'PMTFPS'}
L = [r"% source: data/csv/analysis_ree_samples.csv, analysis_rg_samples.csv",
     r"% generated by generate_checks.ipynb",
     r"\begin{tabular}{llrrrrcc}", r"\toprule",
     r"Chemistry & DP & $n$ & $\langle R_{ee}^2\rangle$ (\AA$^2$) & $\langle R_g^2\rangle$ (\AA$^2$) & $\langle R_{ee}^2\rangle/\langle R_g^2\rangle$ & $\nu$ from $R_{ee}$ & $\nu$ from $R_g$ \\",
     r"\midrule"]
for chem in nus:
    sub = T[T.chem == chem]
    for k, (_, r) in enumerate(sub.iterrows()):
        nuR, sR, nuG, sG = nus[chem]
        c1 = f"{nuR:.3f} $\\pm$ {sR:.3f}" if k == 0 else ""
        c2 = f"{nuG:.3f} $\\pm$ {sG:.3f}" if k == 0 else ""
        L.append(f"{label[chem] if k == 0 else ''} & {r.dp} & {r.n} & {r.R2:.1f} & {r.G2:.1f} & {r.ratio:.2f} & {c1} & {c2} \\\\")
    L.append(r"\midrule" if chem != list(nus)[-1] else r"\bottomrule")
L.append(r"\end{tabular}")
open(os.path.join(TAB_DIR, 'table_chain_stats.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")

In [ ]:
# --- Software versions and random seeds ---
# A static record, not computed (data/derived/software/software_seeds.csv), written as a tabular. The versions of the
# environment running this notebook are printed for comparison with the "Analysis environment" row.
import platform, importlib
SW = pd.read_csv(rev('software', 'software_seeds.csv'), dtype=str, keep_default_na=False)
L = [r"\begin{tabular}{>{\raggedright\arraybackslash}p{3.9cm}>{\raggedright\arraybackslash}p{6.7cm}>{\raggedright\arraybackslash}p{4.5cm}}",
     r"\toprule", r"Item & Version / value & Source \\", r"\midrule"]
for i, (sec, g_) in enumerate(SW.groupby('section', sort=False)):
    if i:
        L.append(r"\midrule")
    L += [f"{r.item} & {r.value} & {r.source} \\\\" for r in g_.itertuples()]
L += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, 'table_software.tex'), 'w', encoding='utf-8').write("\n".join(L) + "\n")
print(f"software table written ({len(SW)} rows)")
print('this environment: Python', platform.python_version(), '|',
      ', '.join(f"{m} {importlib.import_module(m).__version__}" for m in ('numpy', 'scipy', 'pandas', 'matplotlib', 'seaborn', 'networkx')))

In [ ]:
# --- Outputs ---
for d in (TAB_DIR, LOG_DIR, FIG_DIR):
    print(d + '/')
    for f in sorted(os.listdir(d)):
        if os.path.isfile(os.path.join(d, f)):
            print(f"   {f:40s} {os.path.getsize(os.path.join(d, f)):>10,d} bytes")